# NB18 — Functional Landscape PGLS Analysis

**Type:** Exploratory (not pre-registered)  
**Status:** PENDING execution in JupyterHub (Spark required)

**Purpose:** Map β (per-Mb KO density → niche breadth) across 21 KEGG functional categories
to (a) identify a true negative control and (b) place the metal-gene signal in context.

**Design:**
- Part A (6 sets): candidate negative controls (sporulation, secondary metabolism, xenobiotics,
  two-component systems, ABC transporters minus metal KOs, quorum sensing)
- Part B (15 sets): KEGG functional landscape (core metabolism, information processing, AMR)
- Reference: metal Tier 1+2 result (P1, β = −0.021) added as anchor

**Model:** `mean_levins_B_std ~ ko_per_mb_z`, PGLS with Pagel's λ free, GTDB r214 tree,
min n=100 genera per category. BH-FDR across all 21 categories.

**Requires:** JupyterHub (Spark → `kescience_mgnify`)

**Outputs:**
- `data/functional_landscape_results.csv`
- `figures/functional_landscape_forest.png`
- REPORT.md: "Functional landscape analysis" section
- INTERPRETATION_TABLE.md: new exploratory section


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

# Spark setup — identical to NB03 pattern
_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

# Primary PGLS input (for niche breadth and tree matching)
bac_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
trait_df  = bac_base[['genus_lower', 'mean_levins_B_std']].copy()
print(f'Trait table: {len(trait_df)} genera')

# Okabe-Ito palette by group
GROUP_COLORS = {
    'negative_control':       '#E69F00',   # orange
    'core_metabolism':        '#0072B2',   # blue
    'information_processing': '#009E73',   # green
    'metal_related':          '#D55E00',   # red
    'metal_reference':        '#CC79A7',   # pink — the P1 reference bar
}
GROUP_LABELS = {
    'negative_control':       'Candidate negative control (Part A)',
    'core_metabolism':        'Core metabolism (Part B)',
    'information_processing': 'Information processing (Part B)',
    'metal_related':          'Antimicrobial resistance (Part B)',
    'metal_reference':        'Metal genes Tier 1+2 (reference P1)',
}


In [2]:
KEGG_CATEGORIES = {
    # Part A | negative_control | ko04111
    'sporulation': ['K02178', 'K02179', 'K02180', 'K02181', 'K02209', 'K02210', 'K02212', 'K02213', 'K02214', 'K02216', 'K02219', 'K02220', 'K02307', 'K02309', 'K02365', 'K02515', 'K02516', 'K02537', 'K02540', 'K02541', 'K02542', 'K02543', 'K02544', 'K02555', 'K02603', 'K02604', 'K02605', 'K02606', 'K02607', 'K02608', 'K02627', 'K02646', 'K02830', 'K02831', 'K03085', 'K03094', 'K03114', 'K03347', 'K03348', 'K03349', 'K03350', 'K03351', 'K03352', 'K03353', 'K03354', 'K03355', 'K03356', 'K03357', 'K03358', 'K03359', 'K03360', 'K03361', 'K03363', 'K03364', 'K03456', 'K03868', 'K04354', 'K04371', 'K04382', 'K04563', 'K06628', 'K06636', 'K06639', 'K06641', 'K06646', 'K06647', 'K06648', 'K06649', 'K06650', 'K06651', 'K06652', 'K06653', 'K06654', 'K06655', 'K06656', 'K06657', 'K06658', 'K06659', 'K06660', 'K06661', 'K06662', 'K06663', 'K06665', 'K06666', 'K06667', 'K06668', 'K06669', 'K06670', 'K06671', 'K06672', 'K06673', 'K06674', 'K06675', 'K06676', 'K06677', 'K06678', 'K06679', 'K06680', 'K06681', 'K06682', 'K06683', 'K06684', 'K06685', 'K06686', 'K06687', 'K08866', 'K09173', 'K09202', 'K10259', 'K11272', 'K12411', 'K12412', 'K12413', 'K12414', 'K12416', 'K12575', 'K12576', 'K12577', 'K12578', 'K12760', 'K18639', 'K18669', 'K18679', 'K18680', 'K23458', 'K23469'],
    # Part A | negative_control | ko01110
    'secondary_metab': ['K00001', 'K00002', 'K00003', 'K00004', 'K00005', 'K00006', 'K00010', 'K00011', 'K00013', 'K00014', 'K00015', 'K00016', 'K00018', 'K00021', 'K00022', 'K00024', 'K00025', 'K00026', 'K00030', 'K00031', 'K00033', 'K00036', 'K00049', 'K00052', 'K00053', 'K00054', 'K00057', 'K00058', 'K00059', 'K00064', 'K00067', 'K00077', 'K00082', 'K00083', 'K00088', 'K00090', 'K00096', 'K00099', 'K00104', 'K00105', 'K00106', 'K00111', 'K00112', 'K00113', 'K00114', 'K00115', 'K00116', 'K00117', 'K00121', 'K00128', 'K00129', 'K00133', 'K00134', 'K00138', 'K00143', 'K00145', 'K00147', 'K00149', 'K00150', 'K00161', 'K00162', 'K00163', 'K00164', 'K00166', 'K00167', 'K00169', 'K00170', 'K00171', 'K00172', 'K00174', 'K00175', 'K00176', 'K00177', 'K00189', 'K00208', 'K00209', 'K00211', 'K00213', 'K00214', 'K00215', 'K00216', 'K00218', 'K00220', 'K00222', 'K00223', 'K00224', 'K00225', 'K00227', 'K00228', 'K00230', 'K00231', 'K00232', 'K00233', 'K00234', 'K00235', 'K00236', 'K00237', 'K00239', 'K00240', 'K00241', 'K00242', 'K00244', 'K00245', 'K00246', 'K00247', 'K00248', 'K00249', 'K00252', 'K00263', 'K00264', 'K00265', 'K00266', 'K00270', 'K00271', 'K00273', 'K00274', 'K00276', 'K00279', 'K00281', 'K00282', 'K00283', 'K00286', 'K00290', 'K00293', 'K00307', 'K00318', 'K00355', 'K00382', 'K00422', 'K00430', 'K00435', 'K00454', 'K00475', 'K00487', 'K00491', 'K00495', 'K00501', 'K00505', 'K00510', 'K00511', 'K00514', 'K00544', 'K00547', 'K00548', 'K00549', 'K00550', 'K00551', 'K00559', 'K00568', 'K00570', 'K00587', 'K00588', 'K00589', 'K00591', 'K00600', 'K00601', 'K00602', 'K00605', 'K00606', 'K00611', 'K00615', 'K00616', 'K00618', 'K00619', 'K00620', 'K00622', 'K00626', 'K00627', 'K00629', 'K00631', 'K00632', 'K00635', 'K00640', 'K00641', 'K00643', 'K00645', 'K00651', 'K00655', 'K00658', 'K00660', 'K00679', 'K00688', 'K00693', 'K00695', 'K00696', 'K00697', 'K00699', 'K00700', 'K00703', 'K00705', 'K00750', 'K00760', 'K00764', 'K00765', 'K00766', 'K00769', 'K00787', 'K00789', 'K00791', 'K00793', 'K00794', 'K00795', 'K00797', 'K00800', 'K00801', 'K00804', 'K00805', 'K00806', 'K00808', 'K00811', 'K00812', 'K00813', 'K00815', 'K00816', 'K00817', 'K00818', 'K00819', 'K00821', 'K00825', 'K00826', 'K00827', 'K00830', 'K00831', 'K00832', 'K00835', 'K00836', 'K00838', 'K00841', 'K00844', 'K00845', 'K00847', 'K00850', 'K00851', 'K00861', 'K00863', 'K00864', 'K00865', 'K00869', 'K00872', 'K00873', 'K00886', 'K00891', 'K00895', 'K00901', 'K00918', 'K00919', 'K00926', 'K00927', 'K00928', 'K00930', 'K00931', 'K00938', 'K00939', 'K00940', 'K00944', 'K00948', 'K00953', 'K00955', 'K00956', 'K00957', 'K00958', 'K00963', 'K00966', 'K00971', 'K00973', 'K00975', 'K00981', 'K00991', 'K00993', 'K00997', 'K00998', 'K01004', 'K01006', 'K01007', 'K01046', 'K01047', 'K01053', 'K01054', 'K01057', 'K01058', 'K01059', 'K01060', 'K01068', 'K01075', 'K01079', 'K01080', 'K01081', 'K01084', 'K01086', 'K01087', 'K01089', 'K01091', 'K01092', 'K01114', 'K01115', 'K01176', 'K01177', 'K01187', 'K01188', 'K01193', 'K01194', 'K01195', 'K01196', 'K01200', 'K01203', 'K01208', 'K01214', 'K01236', 'K01239', 'K01252', 'K01424', 'K01425', 'K01434', 'K01438', 'K01457', 'K01476', 'K01478', 'K01480', 'K01490', 'K01492', 'K01496', 'K01497', 'K01498', 'K01509', 'K01510', 'K01513', 'K01521', 'K01523', 'K01568', 'K01575', 'K01579', 'K01580', 'K01581', 'K01582', 'K01583', 'K01584', 'K01585', 'K01586', 'K01587', 'K01588', 'K01589', 'K01590', 'K01592', 'K01593', 'K01596', 'K01597', 'K01599', 'K01601', 'K01602', 'K01609', 'K01610', 'K01611', 'K01613', 'K01616', 'K01620', 'K01622', 'K01623', 'K01624', 'K01626', 'K01637', 'K01638', 'K01641', 'K01647', 'K01648', 'K01649', 'K01652', 'K01653', 'K01655', 'K01656', 'K01657', 'K01658', 'K01659', 'K01661', 'K01662', 'K01663', 'K01675', 'K01676', 'K01677', 'K01678', 'K01679', 'K01681', 'K01682', 'K01687', 'K01689', 'K01692', 'K01693', 'K01694', 'K01695', 'K01696', 'K01697', 'K01698', 'K01702', 'K01703', 'K01704', 'K01705', 'K01710', 'K01713', 'K01714', 'K01715', 'K01719', 'K01723', 'K01733', 'K01735', 'K01736', 'K01738', 'K01739', 'K01742', 'K01749', 'K01750', 'K01752', 'K01754', 'K01755', 'K01756', 'K01757', 'K01758', 'K01760', 'K01762', 'K01770', 'K01772', 'K01774', 'K01778', 'K01782', 'K01783', 'K01785', 'K01790', 'K01792', 'K01803', 'K01807', 'K01808', 'K01809', 'K01810', 'K01814', 'K01817', 'K01823', 'K01824', 'K01825', 'K01828', 'K01834', 'K01835', 'K01837', 'K01840', 'K01841', 'K01845', 'K01850', 'K01851', 'K01852', 'K01853', 'K01858', 'K01859', 'K01885', 'K01895', 'K01899', 'K01900', 'K01902', 'K01903', 'K01904', 'K01911', 'K01913', 'K01914', 'K01915', 'K01918', 'K01919', 'K01923', 'K01933', 'K01940', 'K01941', 'K01945', 'K01946', 'K01952', 'K01953', 'K01954', 'K01955', 'K01956', 'K01957', 'K01961', 'K01962', 'K01963', 'K01964', 'K01965', 'K01966', 'K02078', 'K02160', 'K02203', 'K02204', 'K02257', 'K02259', 'K02291', 'K02292', 'K02293', 'K02294', 'K02302', 'K02303', 'K02304', 'K02361', 'K02362', 'K02363', 'K02364', 'K02437', 'K02439', 'K02446', 'K02492', 'K02495', 'K02496', 'K02500', 'K02501', 'K02502', 'K02523', 'K02548', 'K02549', 'K02551', 'K02552', 'K02566', 'K02626', 'K02858', 'K03179', 'K03181', 'K03182', 'K03183', 'K03184', 'K03185', 'K03186', 'K03334', 'K03340', 'K03366', 'K03378', 'K03403', 'K03404', 'K03405', 'K03428', 'K03526', 'K03527', 'K03621', 'K03737', 'K03781', 'K03782', 'K03783', 'K03784', 'K03785', 'K03786', 'K03787', 'K03794', 'K03809', 'K03816', 'K03823', 'K03841', 'K03856', 'K03894', 'K03895', 'K03896', 'K03897', 'K04022', 'K04034', 'K04035', 'K04036', 'K04037', 'K04038', 'K04039', 'K04040', 'K04041', 'K04072', 'K04092', 'K04093', 'K04120', 'K04121', 'K04122', 'K04123', 'K04124', 'K04125', 'K04126', 'K04127', 'K04128', 'K04339', 'K04340', 'K04341', 'K04342', 'K04486', 'K04516', 'K04517', 'K04518', 'K04714', 'K04781', 'K04782', 'K05277', 'K05278', 'K05279', 'K05280', 'K05281', 'K05282', 'K05342', 'K05343', 'K05349', 'K05350', 'K05353', 'K05354', 'K05355', 'K05356', 'K05357', 'K05358', 'K05359', 'K05369', 'K05370', 'K05371', 'K05375', 'K05525', 'K05551', 'K05552', 'K05553', 'K05554', 'K05555', 'K05556', 'K05597', 'K05602', 'K05821', 'K05822', 'K05823', 'K05824', 'K05825', 'K05828', 'K05829', 'K05830', 'K05878', 'K05887', 'K05894', 'K05901', 'K05906', 'K05917', 'K05928', 'K05933', 'K05942', 'K05953', 'K05954', 'K05955', 'K05957', 'K05992', 'K06001', 'K06013', 'K06044', 'K06045', 'K06116', 'K06117', 'K06118', 'K06119', 'K06125', 'K06126', 'K06127', 'K06134', 'K06208', 'K06209', 'K06443', 'K06444', 'K06859', 'K06863', 'K06892', 'K06900', 'K06928', 'K06981', 'K06998', 'K07008', 'K07024', 'K07029', 'K07094', 'K07145', 'K07215', 'K07226', 'K07382', 'K07384', 'K07385', 'K07404', 'K07405', 'K07409', 'K07419', 'K07508', 'K07509', 'K07511', 'K07513', 'K07514', 'K07515', 'K07748', 'K07750', 'K08074', 'K08081', 'K08099', 'K08100', 'K08101', 'K08233', 'K08241', 'K08242', 'K08243', 'K08246', 'K08248', 'K08249', 'K08261', 'K08289', 'K08295', 'K08591', 'K08658', 'K08680', 'K08683', 'K08693', 'K08695', 'K08730', 'K08973', 'K08977', 'K09128', 'K09459', 'K09460', 'K09478', 'K09483', 'K09587', 'K09588', 'K09589', 'K09590', 'K09591', 'K09699', 'K09753', 'K09754', 'K09755', 'K09756', 'K09757', 'K09827', 'K09828', 'K09829', 'K09831', 'K09832', 'K09833', 'K09834', 'K09835', 'K09836', 'K09837', 'K09838', 'K09839', 'K09840', 'K09841', 'K09842', 'K09843', 'K09844', 'K09845', 'K09846', 'K09847', 'K09879', 'K09881', 'K09913', 'K10027', 'K10046', 'K10047', 'K10106', 'K10150', 'K10156', 'K10187', 'K10203', 'K10205', 'K10206', 'K10208', 'K10209', 'K10210', 'K10211', 'K10212', 'K10226', 'K10244', 'K10245', 'K10246', 'K10247', 'K10248', 'K10249', 'K10250', 'K10251', 'K10258', 'K10525', 'K10526', 'K10527', 'K10528', 'K10529', 'K10536', 'K10566', 'K10703', 'K10705', 'K10717', 'K10757', 'K10760', 'K10775', 'K10814', 'K10815', 'K10816', 'K10817', 'K10852', 'K10960', 'K10977', 'K10978', 'K11067', 'K11155', 'K11160', 'K11175', 'K11176', 'K11188', 'K11258', 'K11262', 'K11263', 'K11333', 'K11334', 'K11335', 'K11336', 'K11337', 'K11358', 'K11381', 'K11410', 'K11472', 'K11473', 'K11517', 'K11529', 'K11532', 'K11540', 'K11541', 'K11608', 'K11609', 'K11610', 'K11611', 'K11645', 'K11646', 'K11751', 'K11752', 'K11753', 'K11755', 'K11778', 'K11782', 'K11783', 'K11784', 'K11785', 'K11787', 'K11788', 'K11808', 'K11812', 'K11813', 'K11818', 'K11819', 'K11820', 'K11821', 'K12047', 'K12073', 'K12153', 'K12154', 'K12156', 'K12250', 'K12251', 'K12298', 'K12316', 'K12317', 'K12338', 'K12355', 'K12356', 'K12406', 'K12407', 'K12420', 'K12428', 'K12437', 'K12447', 'K12451', 'K12467', 'K12501', 'K12502', 'K12503', 'K12504', 'K12505', 'K12506', 'K12524', 'K12525', 'K12526', 'K12570', 'K12628', 'K12629', 'K12630', 'K12631', 'K12632', 'K12633', 'K12634', 'K12635', 'K12636', 'K12637', 'K12638', 'K12639', 'K12640', 'K12643', 'K12644', 'K12645', 'K12657', 'K12659', 'K12673', 'K12674', 'K12675', 'K12676', 'K12677', 'K12692', 'K12693', 'K12694', 'K12695', 'K12696', 'K12697', 'K12698', 'K12699', 'K12701', 'K12702', 'K12703', 'K12704', 'K12705', 'K12707', 'K12708', 'K12709', 'K12710', 'K12711', 'K12712', 'K12713', 'K12714', 'K12719', 'K12720', 'K12721', 'K12722', 'K12723', 'K12724', 'K12729', 'K12730', 'K12731', 'K12742', 'K12743', 'K12744', 'K12745', 'K12747', 'K12748', 'K12901', 'K12902', 'K12903', 'K12904', 'K12905', 'K12906', 'K12907', 'K12908', 'K12909', 'K12910', 'K12911', 'K12912', 'K12913', 'K12914', 'K12915', 'K12917', 'K12918', 'K12919', 'K12920', 'K12921', 'K12923', 'K12924', 'K12926', 'K12927', 'K12928', 'K12929', 'K12930', 'K12934', 'K12935', 'K12936', 'K12939', 'K12957', 'K12972', 'K13027', 'K13029', 'K13030', 'K13031', 'K13032', 'K13033', 'K13034', 'K13035', 'K13037', 'K13051', 'K13063', 'K13064', 'K13065', 'K13066', 'K13067', 'K13068', 'K13070', 'K13071', 'K13077', 'K13079', 'K13081', 'K13082', 'K13083', 'K13222', 'K13223', 'K13224', 'K13225', 'K13226', 'K13227', 'K13229', 'K13230', 'K13231', 'K13232', 'K13233', 'K13234', 'K13235', 'K13240', 'K13241', 'K13242', 'K13257', 'K13258', 'K13259', 'K13260', 'K13261', 'K13262', 'K13263', 'K13265', 'K13266', 'K13267', 'K13269', 'K13272', 'K13273', 'K13306', 'K13307', 'K13308', 'K13309', 'K13310', 'K13311', 'K13312', 'K13313', 'K13315', 'K13316', 'K13317', 'K13318', 'K13319', 'K13320', 'K13322', 'K13326', 'K13327', 'K13328', 'K13329', 'K13330', 'K13332', 'K13356', 'K13371', 'K13372', 'K13373', 'K13382', 'K13383', 'K13384', 'K13385', 'K13386', 'K13387', 'K13389', 'K13390', 'K13391', 'K13392', 'K13393', 'K13394', 'K13395', 'K13396', 'K13397', 'K13398', 'K13400', 'K13401', 'K13427', 'K13492', 'K13493', 'K13494', 'K13495', 'K13496', 'K13497', 'K13498', 'K13501', 'K13503', 'K13506', 'K13507', 'K13508', 'K13509', 'K13513', 'K13517', 'K13519', 'K13523', 'K13534', 'K13542', 'K13543', 'K13544', 'K13545', 'K13546', 'K13547', 'K13548', 'K13549', 'K13550', 'K13551', 'K13552', 'K13553', 'K13554', 'K13555', 'K13556', 'K13557', 'K13559', 'K13560', 'K13561', 'K13562', 'K13563', 'K13564', 'K13565', 'K13574', 'K13600', 'K13601', 'K13602', 'K13603', 'K13604', 'K13605', 'K13606', 'K13607', 'K13608', 'K13644', 'K13679', 'K13713', 'K13727', 'K13745', 'K13774', 'K13775', 'K13787', 'K13789', 'K13799', 'K13810', 'K13811', 'K13821', 'K13829', 'K13830', 'K13832', 'K13853', 'K13937', 'K13951', 'K13952', 'K13953', 'K13954', 'K13979', 'K13997', 'K14028', 'K14029', 'K14036', 'K14037', 'K14038', 'K14039', 'K14040', 'K14041', 'K14042', 'K14043', 'K14044', 'K14045', 'K14046', 'K14047', 'K14066', 'K14073', 'K14074', 'K14075', 'K14076', 'K14085', 'K14130', 'K14132', 'K14134', 'K14135', 'K14152', 'K14155', 'K14157', 'K14163', 'K14170', 'K14173', 'K14174', 'K14176', 'K14177', 'K14178', 'K14179', 'K14180', 'K14182', 'K14183', 'K14184', 'K14186', 'K14187', 'K14190', 'K14215', 'K14244', 'K14245', 'K14246', 'K14249', 'K14250', 'K14251', 'K14252', 'K14253', 'K14254', 'K14255', 'K14256', 'K14257', 'K14260', 'K14266', 'K14271', 'K14272', 'K14329', 'K14366', 'K14367', 'K14368', 'K14369', 'K14370', 'K14371', 'K14372', 'K14373', 'K14374', 'K14375', 'K14423', 'K14424', 'K14452', 'K14454', 'K14455', 'K14456', 'K14471', 'K14472', 'K14541', 'K14577', 'K14593', 'K14594', 'K14595', 'K14596', 'K14597', 'K14598', 'K14605', 'K14606', 'K14621', 'K14626', 'K14627', 'K14628', 'K14629', 'K14630', 'K14631', 'K14632', 'K14633', 'K14641', 'K14642', 'K14652', 'K14656', 'K14674', 'K14675', 'K14677', 'K14681', 'K14682', 'K14759', 'K14760', 'K14975', 'K14976', 'K14984', 'K15036', 'K15037', 'K15086', 'K15087', 'K15088', 'K15089', 'K15090', 'K15091', 'K15092', 'K15093', 'K15094', 'K15095', 'K15096', 'K15097', 'K15098', 'K15099', 'K15226', 'K15227', 'K15230', 'K15231', 'K15314', 'K15315', 'K15316', 'K15320', 'K15397', 'K15404', 'K15405', 'K15406', 'K15467', 'K15472', 'K15506', 'K15633', 'K15634', 'K15635', 'K15639', 'K15652', 'K15728', 'K15741', 'K15742', 'K15744', 'K15745', 'K15746', 'K15747', 'K15748', 'K15759', 'K15774', 'K15775', 'K15776', 'K15777', 'K15778', 'K15779', 'K15780', 'K15791', 'K15793', 'K15800', 'K15808', 'K15812', 'K15813', 'K15814', 'K15815', 'K15816', 'K15817', 'K15819', 'K15821', 'K15823', 'K15849', 'K15860', 'K15884', 'K15885', 'K15886', 'K15887', 'K15889', 'K15890', 'K15891', 'K15892', 'K15893', 'K15907', 'K15916', 'K15918', 'K15919', 'K15926', 'K15927', 'K15928', 'K15929', 'K15930', 'K15931', 'K15932', 'K15933', 'K15934', 'K15935', 'K15936', 'K15937', 'K15938', 'K15939', 'K15941', 'K15942', 'K15943', 'K15944', 'K15945', 'K15946', 'K15947', 'K15948', 'K15949', 'K15950', 'K15951', 'K15952', 'K15953', 'K15954', 'K15955', 'K15956', 'K15957', 'K15958', 'K15959', 'K15960', 'K15961', 'K15962', 'K15963', 'K15964', 'K15965', 'K15966', 'K15967', 'K15968', 'K15969', 'K15970', 'K15971', 'K15972', 'K15988', 'K15989', 'K15990', 'K15991', 'K15992', 'K15993', 'K15994', 'K15995', 'K15996', 'K15997', 'K15998', 'K15999', 'K16000', 'K16001', 'K16002', 'K16003', 'K16004', 'K16005', 'K16006', 'K16007', 'K16008', 'K16009', 'K16010', 'K16011', 'K16015', 'K16016', 'K16017', 'K16018', 'K16019', 'K16020', 'K16021', 'K16023', 'K16033', 'K16034', 'K16035', 'K16036', 'K16037', 'K16038', 'K16039', 'K16040', 'K16055', 'K16082', 'K16083', 'K16084', 'K16085', 'K16086', 'K16149', 'K16150', 'K16153', 'K16207', 'K16265', 'K16266', 'K16305', 'K16306', 'K16339', 'K16342', 'K16343', 'K16368', 'K16369', 'K16370', 'K16421', 'K16422', 'K16423', 'K16424', 'K16425', 'K16426', 'K16427', 'K16431', 'K16435', 'K16436', 'K16437', 'K16438', 'K16439', 'K16619', 'K16792', 'K16793', 'K16816', 'K16817', 'K16818', 'K16860', 'K16881', 'K17054', 'K17055', 'K17056', 'K17058', 'K17059', 'K17069', 'K17078', 'K17103', 'K17104', 'K17105', 'K17194', 'K17211', 'K17212', 'K17217', 'K17360', 'K17450', 'K17475', 'K17476', 'K17497', 'K17625', 'K17626', 'K17643', 'K17644', 'K17645', 'K17646', 'K17647', 'K17648', 'K17649', 'K17650', 'K17651', 'K17652', 'K17717', 'K17744', 'K17746', 'K17747', 'K17753', 'K17819', 'K17825', 'K17826', 'K17827', 'K17829', 'K17830', 'K17835', 'K17836', 'K17841', 'K17842', 'K17872', 'K17876', 'K17900', 'K17911', 'K17912', 'K17913', 'K17940', 'K17942', 'K17947', 'K17961', 'K17982', 'K17989', 'K18000', 'K18001', 'K18002', 'K18003', 'K18009', 'K18010', 'K18054', 'K18056', 'K18057', 'K18062', 'K18091', 'K18097', 'K18108', 'K18113', 'K18118', 'K18121', 'K18124', 'K18125', 'K18209', 'K18210', 'K18221', 'K18240', 'K18279', 'K18280', 'K18281', 'K18284', 'K18285', 'K18286', 'K18287', 'K18315', 'K18316', 'K18317', 'K18318', 'K18319', 'K18368', 'K18383', 'K18385', 'K18386', 'K18387', 'K18388', 'K18389', 'K18390', 'K18391', 'K18392', 'K18393', 'K18394', 'K18395', 'K18396', 'K18397', 'K18447', 'K18472', 'K18532', 'K18533', 'K18534', 'K18537', 'K18556', 'K18558', 'K18562', 'K18563', 'K18565', 'K18566', 'K18568', 'K18569', 'K18570', 'K18571', 'K18572', 'K18582', 'K18583', 'K18586', 'K18606', 'K18649', 'K18652', 'K18653', 'K18654', 'K18686', 'K18689', 'K18690', 'K18693', 'K18800', 'K18836', 'K18851', 'K18857', 'K18858', 'K18859', 'K18860', 'K18884', 'K18911', 'K18912', 'K18913', 'K18933', 'K18966', 'K19007', 'K19064', 'K19073', 'K19102', 'K19103', 'K19104', 'K19105', 'K19106', 'K19107', 'K19108', 'K19109', 'K19110', 'K19111', 'K19112', 'K19113', 'K19177', 'K19180', 'K19182', 'K19183', 'K19184', 'K19200', 'K19222', 'K19243', 'K19267', 'K19269', 'K19312', 'K19517', 'K19532', 'K19546', 'K19547', 'K19548', 'K19549', 'K19550', 'K19566', 'K19567', 'K19568', 'K19569', 'K19570', 'K19571', 'K19650', 'K19651', 'K19652', 'K19664', 'K19665', 'K19698', 'K19723', 'K19724', 'K19725', 'K19726', 'K19727', 'K19741', 'K19813', 'K19834', 'K19835', 'K19836', 'K19853', 'K19854', 'K19855', 'K19856', 'K19857', 'K19858', 'K19859', 'K19884', 'K19885', 'K19886', 'K19887', 'K19888', 'K19889', 'K19969', 'K19970', 'K19974', 'K19978', 'K19979', 'K19981', 'K19982', 'K20039', 'K20075', 'K20076', 'K20077', 'K20078', 'K20079', 'K20080', 'K20081', 'K20082', 'K20085', 'K20086', 'K20087', 'K20088', 'K20089', 'K20090', 'K20142', 'K20144', 'K20152', 'K20156', 'K20159', 'K20204', 'K20246', 'K20247', 'K20257', 'K20260', 'K20261', 'K20262', 'K20420', 'K20421', 'K20422', 'K20423', 'K20424', 'K20425', 'K20426', 'K20427', 'K20428', 'K20430', 'K20431', 'K20432', 'K20433', 'K20434', 'K20435', 'K20436', 'K20437', 'K20438', 'K20439', 'K20440', 'K20441', 'K20442', 'K20443', 'K20501', 'K20502', 'K20503', 'K20508', 'K20512', 'K20513', 'K20514', 'K20515', 'K20516', 'K20517', 'K20518', 'K20519', 'K20545', 'K20546', 'K20561', 'K20565', 'K20566', 'K20567', 'K20568', 'K20569', 'K20570', 'K20571', 'K20572', 'K20573', 'K20574', 'K20575', 'K20576', 'K20577', 'K20578', 'K20579', 'K20580', 'K20581', 'K20582', 'K20583', 'K20584', 'K20585', 'K20586', 'K20587', 'K20588', 'K20589', 'K20590', 'K20591', 'K20592', 'K20593', 'K20594', 'K20595', 'K20596', 'K20597', 'K20598', 'K20611', 'K20616', 'K20618', 'K20621', 'K20623', 'K20657', 'K20658', 'K20659', 'K20666', 'K20667', 'K20678', 'K20679', 'K20680', 'K20681', 'K20682', 'K20709', 'K20771', 'K20772', 'K20802', 'K20810', 'K20812', 'K20860', 'K20861', 'K20862', 'K20884', 'K20930', 'K20940', 'K20979', 'K20980', 'K20981', 'K20982', 'K20983', 'K20984', 'K20985', 'K20986', 'K21026', 'K21036', 'K21058', 'K21063', 'K21064', 'K21069', 'K21070', 'K21071', 'K21103', 'K21120', 'K21146', 'K21160', 'K21161', 'K21162', 'K21163', 'K21164', 'K21165', 'K21166', 'K21167', 'K21168', 'K21169', 'K21170', 'K21171', 'K21172', 'K21173', 'K21174', 'K21175', 'K21176', 'K21177', 'K21178', 'K21179', 'K21181', 'K21182', 'K21184', 'K21185', 'K21188', 'K21191', 'K21192', 'K21202', 'K21203', 'K21204', 'K21205', 'K21207', 'K21208', 'K21210', 'K21211', 'K21212', 'K21213', 'K21214', 'K21215', 'K21222', 'K21223', 'K21224', 'K21225', 'K21227', 'K21228', 'K21254', 'K21255', 'K21256', 'K21257', 'K21258', 'K21259', 'K21260', 'K21261', 'K21262', 'K21263', 'K21268', 'K21271', 'K21272', 'K21273', 'K21274', 'K21275', 'K21291', 'K21292', 'K21294', 'K21295', 'K21297', 'K21301', 'K21311', 'K21312', 'K21325', 'K21326', 'K21327', 'K21328', 'K21329', 'K21330', 'K21331', 'K21332', 'K21333', 'K21335', 'K21336', 'K21337', 'K21338', 'K21346', 'K21354', 'K21359', 'K21360', 'K21371', 'K21372', 'K21373', 'K21374', 'K21383', 'K21418', 'K21428', 'K21477', 'K21480', 'K21513', 'K21539', 'K21540', 'K21550', 'K21568', 'K21580', 'K21588', 'K21610', 'K21611', 'K21612', 'K21692', 'K21693', 'K21714', 'K21715', 'K21718', 'K21719', 'K21721', 'K21722', 'K21723', 'K21724', 'K21778', 'K21779', 'K21780', 'K21781', 'K21782', 'K21783', 'K21784', 'K21785', 'K21786', 'K21787', 'K21788', 'K21789', 'K21790', 'K21791', 'K21792', 'K21793', 'K21796', 'K21819', 'K21895', 'K21896', 'K21898', 'K21925', 'K21926', 'K21927', 'K21928', 'K21937', 'K21938', 'K21949', 'K22011', 'K22012', 'K22013', 'K22049', 'K22050', 'K22064', 'K22065', 'K22088', 'K22089', 'K22090', 'K22091', 'K22092', 'K22093', 'K22094', 'K22095', 'K22096', 'K22097', 'K22098', 'K22113', 'K22114', 'K22223', 'K22225', 'K22226', 'K22227', 'K22251', 'K22269', 'K22283', 'K22284', 'K22305', 'K22321', 'K22323', 'K22326', 'K22327', 'K22337', 'K22365', 'K22374', 'K22389', 'K22392', 'K22395', 'K22433', 'K22434', 'K22435', 'K22436', 'K22440', 'K22445', 'K22450', 'K22451', 'K22458', 'K22473', 'K22474', 'K22476', 'K22477', 'K22478', 'K22492', 'K22502', 'K22554', 'K22568', 'K22569', 'K22570', 'K22571', 'K22572', 'K22573', 'K22574', 'K22575', 'K22588', 'K22598', 'K22634', 'K22635', 'K22636', 'K22637', 'K22638', 'K22639', 'K22697', 'K22706', 'K22772', 'K22794', 'K22798', 'K22799', 'K22800', 'K22801', 'K22802', 'K22813', 'K22831', 'K22842', 'K22845', 'K22848', 'K22849', 'K22912', 'K22932', 'K22934', 'K22945', 'K22946', 'K22947', 'K22948', 'K22949', 'K22982', 'K22997', 'K22998', 'K23037', 'K23053', 'K23094', 'K23095', 'K23109', 'K23136', 'K23137', 'K23144', 'K23145', 'K23157', 'K23158', 'K23179', 'K23180', 'K23232', 'K23260', 'K23264', 'K23265', 'K23269', 'K23270', 'K23276', 'K23304', 'K23371', 'K23372', 'K23373', 'K23374', 'K23375', 'K23378', 'K23446', 'K23447', 'K23452', 'K23485', 'K23493', 'K23494', 'K23520', 'K23521', 'K23522', 'K23523', 'K23524', 'K23558', 'K23646', 'K23647', 'K23662', 'K23665', 'K23666', 'K23667', 'K23668', 'K23669', 'K23670', 'K23671', 'K23672', 'K23673', 'K23736', 'K23737', 'K23810', 'K23825', 'K23862', 'K23888', 'K23889', 'K23890', 'K23891', 'K23892', 'K23987', 'K23989', 'K23990', 'K23999', 'K24000', 'K24017', 'K24018', 'K24034', 'K24041', 'K24042', 'K24108', 'K24109', 'K24110', 'K24111', 'K24112', 'K24182', 'K24189', 'K24263', 'K24264', 'K24276', 'K24277', 'K24287', 'K24292', 'K24390', 'K24438', 'K24439', 'K24440', 'K24441', 'K24460', 'K24528', 'K24529', 'K24530', 'K24531', 'K24541', 'K24843', 'K24844', 'K24845', 'K24855', 'K24856', 'K24857', 'K24858', 'K24866', 'K24867', 'K24872', 'K24873', 'K25026', 'K25031', 'K25033', 'K25060', 'K25072', 'K25073', 'K25074', 'K25075', 'K25463', 'K25486', 'K25487', 'K25488', 'K25491', 'K25517', 'K25518', 'K25522', 'K25523', 'K25528', 'K25562', 'K25563', 'K25580', 'K25581', 'K25582', 'K25583', 'K25584', 'K25594', 'K25597', 'K25598', 'K25801', 'K25824', 'K25959', 'K25995', 'K25996', 'K25998', 'K26037', 'K26038', 'K26318', 'K26400', 'K26481', 'K26482', 'K26483', 'K26484', 'K26485', 'K26487', 'K26488', 'K26489', 'K26490', 'K26491', 'K26493', 'K26494', 'K26495', 'K27070', 'K27071', 'K27072', 'K27073', 'K27074', 'K27093', 'K27094', 'K27095', 'K27306', 'K27308', 'K27359', 'K27394', 'K27501', 'K27502', 'K27503', 'K27538', 'K27539', 'K27542', 'K27545', 'K27598', 'K27602', 'K27636', 'K27675', 'K27683', 'K27684', 'K27689', 'K27776', 'K27777', 'K27778', 'K27779', 'K27780', 'K27781', 'K27782', 'K27783', 'K27784', 'K27785', 'K27786', 'K27787', 'K27788', 'K27789', 'K27790', 'K27791', 'K27792', 'K27793', 'K27794', 'K27795', 'K27796', 'K27797', 'K27802', 'K27839', 'K27840', 'K27844', 'K27849', 'K27850', 'K27857', 'K27859', 'K27860', 'K28013', 'K28034', 'K28131', 'K28138', 'K28139', 'K28140', 'K28205', 'K28301', 'K28316', 'K28318', 'K28328', 'K28354', 'K28367', 'K28368', 'K28384', 'K28464', 'K28465', 'K28472', 'K28477', 'K28482', 'K28535', 'K28536', 'K28537', 'K28615', 'K28660', 'K28682', 'K28683', 'K28684', 'K28694', 'K28695', 'K28696', 'K28697', 'K28698', 'K28699', 'K28700', 'K28702', 'K28703', 'K28704', 'K28705', 'K28706', 'K28707', 'K28708', 'K28751', 'K28799', 'K28801', 'K28825', 'K28842', 'K28849', 'K28898', 'K28899', 'K28924', 'K28982', 'K28987', 'K28988'],
    # Part A | negative_control | ko01120
    'xenobiotics': ['K00001', 'K00002', 'K00003', 'K00010', 'K00012', 'K00015', 'K00016', 'K00018', 'K00022', 'K00023', 'K00024', 'K00025', 'K00026', 'K00028', 'K00029', 'K00030', 'K00031', 'K00032', 'K00033', 'K00034', 'K00036', 'K00040', 'K00041', 'K00048', 'K00049', 'K00051', 'K00055', 'K00058', 'K00064', 'K00065', 'K00073', 'K00074', 'K00087', 'K00090', 'K00093', 'K00100', 'K00104', 'K00106', 'K00114', 'K00116', 'K00121', 'K00122', 'K00123', 'K00124', 'K00125', 'K00126', 'K00127', 'K00128', 'K00129', 'K00131', 'K00132', 'K00133', 'K00134', 'K00135', 'K00137', 'K00138', 'K00139', 'K00140', 'K00141', 'K00146', 'K00148', 'K00149', 'K00150', 'K00151', 'K00152', 'K00157', 'K00161', 'K00162', 'K00163', 'K00164', 'K00169', 'K00170', 'K00171', 'K00172', 'K00174', 'K00175', 'K00176', 'K00177', 'K00184', 'K00185', 'K00189', 'K00192', 'K00193', 'K00194', 'K00195', 'K00196', 'K00197', 'K00198', 'K00200', 'K00201', 'K00202', 'K00203', 'K00204', 'K00205', 'K00206', 'K00209', 'K00215', 'K00217', 'K00233', 'K00234', 'K00235', 'K00236', 'K00237', 'K00239', 'K00240', 'K00241', 'K00242', 'K00244', 'K00245', 'K00246', 'K00247', 'K00248', 'K00252', 'K00260', 'K00261', 'K00262', 'K00264', 'K00265', 'K00266', 'K00275', 'K00284', 'K00288', 'K00297', 'K00299', 'K00300', 'K00306', 'K00317', 'K00319', 'K00320', 'K00360', 'K00361', 'K00362', 'K00363', 'K00365', 'K00366', 'K00367', 'K00368', 'K00370', 'K00371', 'K00372', 'K00374', 'K00376', 'K00380', 'K00381', 'K00382', 'K00385', 'K00386', 'K00387', 'K00390', 'K00392', 'K00394', 'K00395', 'K00399', 'K00400', 'K00401', 'K00402', 'K00440', 'K00441', 'K00442', 'K00443', 'K00446', 'K00448', 'K00449', 'K00450', 'K00451', 'K00455', 'K00462', 'K00468', 'K00480', 'K00481', 'K00483', 'K00484', 'K00492', 'K00529', 'K00531', 'K00539', 'K00577', 'K00578', 'K00579', 'K00580', 'K00581', 'K00582', 'K00583', 'K00584', 'K00589', 'K00600', 'K00615', 'K00616', 'K00622', 'K00625', 'K00626', 'K00627', 'K00632', 'K00640', 'K00641', 'K00651', 'K00656', 'K00658', 'K00672', 'K00674', 'K00814', 'K00821', 'K00823', 'K00824', 'K00825', 'K00830', 'K00831', 'K00836', 'K00844', 'K00845', 'K00846', 'K00848', 'K00850', 'K00851', 'K00855', 'K00860', 'K00863', 'K00865', 'K00872', 'K00873', 'K00874', 'K00879', 'K00881', 'K00886', 'K00895', 'K00918', 'K00925', 'K00926', 'K00927', 'K00928', 'K00948', 'K00955', 'K00956', 'K00957', 'K00958', 'K00963', 'K00988', 'K01006', 'K01007', 'K01011', 'K01026', 'K01031', 'K01032', 'K01034', 'K01035', 'K01039', 'K01040', 'K01041', 'K01051', 'K01053', 'K01055', 'K01057', 'K01061', 'K01070', 'K01075', 'K01079', 'K01082', 'K01085', 'K01086', 'K01100', 'K01101', 'K01184', 'K01213', 'K01426', 'K01427', 'K01428', 'K01429', 'K01430', 'K01439', 'K01453', 'K01457', 'K01466', 'K01471', 'K01477', 'K01491', 'K01499', 'K01501', 'K01502', 'K01506', 'K01512', 'K01555', 'K01557', 'K01560', 'K01561', 'K01563', 'K01576', 'K01580', 'K01582', 'K01586', 'K01595', 'K01596', 'K01601', 'K01602', 'K01607', 'K01610', 'K01612', 'K01615', 'K01616', 'K01617', 'K01620', 'K01621', 'K01622', 'K01623', 'K01624', 'K01625', 'K01628', 'K01629', 'K01637', 'K01638', 'K01640', 'K01647', 'K01648', 'K01655', 'K01659', 'K01666', 'K01670', 'K01675', 'K01676', 'K01677', 'K01678', 'K01679', 'K01681', 'K01682', 'K01684', 'K01685', 'K01686', 'K01689', 'K01690', 'K01692', 'K01698', 'K01705', 'K01714', 'K01719', 'K01720', 'K01721', 'K01728', 'K01730', 'K01731', 'K01733', 'K01734', 'K01738', 'K01739', 'K01749', 'K01774', 'K01778', 'K01781', 'K01782', 'K01783', 'K01785', 'K01792', 'K01796', 'K01799', 'K01800', 'K01801', 'K01803', 'K01807', 'K01808', 'K01810', 'K01812', 'K01813', 'K01815', 'K01818', 'K01820', 'K01821', 'K01822', 'K01825', 'K01826', 'K01834', 'K01835', 'K01837', 'K01841', 'K01845', 'K01846', 'K01847', 'K01848', 'K01849', 'K01856', 'K01857', 'K01860', 'K01865', 'K01885', 'K01895', 'K01899', 'K01900', 'K01902', 'K01903', 'K01905', 'K01912', 'K01913', 'K01915', 'K01938', 'K01941', 'K01946', 'K01948', 'K01958', 'K01959', 'K01960', 'K01961', 'K01962', 'K01963', 'K01964', 'K01965', 'K01966', 'K02083', 'K02160', 'K02203', 'K02204', 'K02302', 'K02303', 'K02304', 'K02305', 'K02381', 'K02439', 'K02446', 'K02492', 'K02496', 'K02509', 'K02510', 'K02554', 'K02567', 'K02568', 'K02586', 'K02588', 'K02591', 'K02609', 'K02610', 'K02611', 'K02612', 'K02613', 'K02615', 'K02618', 'K02768', 'K02769', 'K02770', 'K02821', 'K02822', 'K03077', 'K03078', 'K03079', 'K03186', 'K03268', 'K03333', 'K03335', 'K03336', 'K03337', 'K03338', 'K03339', 'K03379', 'K03380', 'K03381', 'K03382', 'K03383', 'K03385', 'K03388', 'K03389', 'K03390', 'K03391', 'K03396', 'K03417', 'K03421', 'K03422', 'K03430', 'K03464', 'K03475', 'K03476', 'K03532', 'K03533', 'K03737', 'K03738', 'K03777', 'K03778', 'K03841', 'K03862', 'K03863', 'K03894', 'K03895', 'K03896', 'K03897', 'K03972', 'K04021', 'K04022', 'K04041', 'K04072', 'K04073', 'K04091', 'K04098', 'K04099', 'K04100', 'K04101', 'K04102', 'K04105', 'K04107', 'K04108', 'K04109', 'K04110', 'K04112', 'K04113', 'K04114', 'K04115', 'K04116', 'K04117', 'K04118', 'K04480', 'K04561', 'K04755', 'K04835', 'K05275', 'K05296', 'K05298', 'K05299', 'K05301', 'K05306', 'K05308', 'K05394', 'K05523', 'K05549', 'K05550', 'K05599', 'K05600', 'K05606', 'K05708', 'K05709', 'K05710', 'K05711', 'K05712', 'K05713', 'K05714', 'K05783', 'K05784', 'K05797', 'K05824', 'K05884', 'K05898', 'K05907', 'K05908', 'K05913', 'K05921', 'K05942', 'K05979', 'K06034', 'K06035', 'K06151', 'K06152', 'K06281', 'K06282', 'K06446', 'K06606', 'K06718', 'K06720', 'K06859', 'K06881', 'K06912', 'K07046', 'K07104', 'K07127', 'K07248', 'K07250', 'K07306', 'K07307', 'K07308', 'K07381', 'K07404', 'K07511', 'K07514', 'K07515', 'K07516', 'K07534', 'K07535', 'K07536', 'K07537', 'K07538', 'K07539', 'K07540', 'K07543', 'K07544', 'K07545', 'K07546', 'K07547', 'K07548', 'K07549', 'K07550', 'K07811', 'K07812', 'K07821', 'K07823', 'K07824', 'K08074', 'K08093', 'K08094', 'K08097', 'K08264', 'K08265', 'K08295', 'K08323', 'K08324', 'K08352', 'K08353', 'K08354', 'K08357', 'K08358', 'K08359', 'K08685', 'K08686', 'K08689', 'K08690', 'K08691', 'K08692', 'K08710', 'K08726', 'K09251', 'K09459', 'K09461', 'K09709', 'K09788', 'K10150', 'K10215', 'K10216', 'K10217', 'K10218', 'K10219', 'K10220', 'K10221', 'K10222', 'K10437', 'K10438', 'K10533', 'K10534', 'K10535', 'K10566', 'K10616', 'K10617', 'K10618', 'K10619', 'K10620', 'K10621', 'K10622', 'K10623', 'K10674', 'K10676', 'K10678', 'K10679', 'K10680', 'K10700', 'K10701', 'K10702', 'K10713', 'K10714', 'K10764', 'K10797', 'K10944', 'K10945', 'K10946', 'K10977', 'K10978', 'K11177', 'K11178', 'K11180', 'K11181', 'K11212', 'K11260', 'K11261', 'K11262', 'K11263', 'K11311', 'K11389', 'K11395', 'K11441', 'K11472', 'K11473', 'K11517', 'K11529', 'K11532', 'K11645', 'K11731', 'K11779', 'K11780', 'K11781', 'K11943', 'K11944', 'K11945', 'K11946', 'K11947', 'K11948', 'K11949', 'K12234', 'K12407', 'K12447', 'K12466', 'K12524', 'K12525', 'K12526', 'K12660', 'K12661', 'K12730', 'K12731', 'K12957', 'K12972', 'K13034', 'K13039', 'K13479', 'K13480', 'K13481', 'K13482', 'K13483', 'K13524', 'K13542', 'K13543', 'K13609', 'K13745', 'K13767', 'K13774', 'K13775', 'K13776', 'K13777', 'K13778', 'K13779', 'K13788', 'K13810', 'K13811', 'K13812', 'K13831', 'K13942', 'K13953', 'K13954', 'K13979', 'K13995', 'K13997', 'K14028', 'K14029', 'K14048', 'K14067', 'K14080', 'K14081', 'K14082', 'K14083', 'K14084', 'K14085', 'K14126', 'K14127', 'K14128', 'K14138', 'K14163', 'K14267', 'K14268', 'K14272', 'K14333', 'K14334', 'K14338', 'K14417', 'K14418', 'K14419', 'K14420', 'K14421', 'K14422', 'K14446', 'K14447', 'K14448', 'K14449', 'K14451', 'K14454', 'K14455', 'K14465', 'K14466', 'K14467', 'K14468', 'K14469', 'K14470', 'K14471', 'K14472', 'K14481', 'K14482', 'K14519', 'K14520', 'K14534', 'K14541', 'K14578', 'K14579', 'K14580', 'K14581', 'K14582', 'K14583', 'K14584', 'K14585', 'K14586', 'K14599', 'K14600', 'K14601', 'K14602', 'K14603', 'K14604', 'K14727', 'K14730', 'K14731', 'K14732', 'K14733', 'K14746', 'K14747', 'K14748', 'K14749', 'K14751', 'K14940', 'K14941', 'K14974', 'K14977', 'K15016', 'K15017', 'K15018', 'K15019', 'K15020', 'K15022', 'K15023', 'K15024', 'K15036', 'K15037', 'K15038', 'K15039', 'K15052', 'K15054', 'K15055', 'K15056', 'K15057', 'K15058', 'K15059', 'K15060', 'K15061', 'K15062', 'K15063', 'K15064', 'K15065', 'K15066', 'K15228', 'K15229', 'K15230', 'K15231', 'K15232', 'K15233', 'K15234', 'K15236', 'K15237', 'K15238', 'K15239', 'K15240', 'K15241', 'K15242', 'K15243', 'K15244', 'K15245', 'K15246', 'K15253', 'K15357', 'K15358', 'K15359', 'K15371', 'K15422', 'K15481', 'K15511', 'K15512', 'K15514', 'K15567', 'K15568', 'K15569', 'K15570', 'K15571', 'K15572', 'K15574', 'K15575', 'K15633', 'K15634', 'K15635', 'K15736', 'K15737', 'K15749', 'K15750', 'K15751', 'K15752', 'K15753', 'K15754', 'K15755', 'K15756', 'K15757', 'K15758', 'K15759', 'K15760', 'K15761', 'K15762', 'K15763', 'K15764', 'K15765', 'K15766', 'K15767', 'K15768', 'K15769', 'K15778', 'K15779', 'K15783', 'K15784', 'K15785', 'K15786', 'K15860', 'K15864', 'K15866', 'K15876', 'K15877', 'K15887', 'K15889', 'K15893', 'K15916', 'K15919', 'K15981', 'K15982', 'K15983', 'K16043', 'K16044', 'K16045', 'K16046', 'K16047', 'K16049', 'K16050', 'K16051', 'K16157', 'K16158', 'K16159', 'K16160', 'K16161', 'K16162', 'K16163', 'K16164', 'K16165', 'K16171', 'K16173', 'K16176', 'K16177', 'K16178', 'K16179', 'K16180', 'K16181', 'K16182', 'K16190', 'K16239', 'K16242', 'K16243', 'K16244', 'K16245', 'K16246', 'K16249', 'K16254', 'K16255', 'K16256', 'K16257', 'K16258', 'K16259', 'K16260', 'K16268', 'K16269', 'K16303', 'K16304', 'K16305', 'K16306', 'K16319', 'K16320', 'K16370', 'K16514', 'K16792', 'K16793', 'K16838', 'K16839', 'K16840', 'K16841', 'K16842', 'K16849', 'K16850', 'K16871', 'K16873', 'K16874', 'K16875', 'K16876', 'K16877', 'K16878', 'K16879', 'K16880', 'K16901', 'K16902', 'K16936', 'K16937', 'K16950', 'K16951', 'K16952', 'K16953', 'K16964', 'K16965', 'K16966', 'K16967', 'K16968', 'K16969', 'K17048', 'K17049', 'K17066', 'K17067', 'K17068', 'K17069', 'K17070', 'K17100', 'K17195', 'K17218', 'K17219', 'K17220', 'K17221', 'K17222', 'K17223', 'K17224', 'K17225', 'K17226', 'K17227', 'K17228', 'K17229', 'K17230', 'K17285', 'K17450', 'K17463', 'K17464', 'K17465', 'K17466', 'K17467', 'K17468', 'K17486', 'K17725', 'K17753', 'K17829', 'K17832', 'K17865', 'K17877', 'K17993', 'K17994', 'K17995', 'K17996', 'K18020', 'K18021', 'K18022', 'K18028', 'K18029', 'K18030', 'K18067', 'K18068', 'K18069', 'K18071', 'K18074', 'K18075', 'K18076', 'K18077', 'K18087', 'K18088', 'K18089', 'K18090', 'K18092', 'K18102', 'K18103', 'K18106', 'K18107', 'K18118', 'K18121', 'K18126', 'K18127', 'K18128', 'K18151', 'K18199', 'K18209', 'K18210', 'K18227', 'K18242', 'K18243', 'K18248', 'K18249', 'K18251', 'K18252', 'K18253', 'K18254', 'K18255', 'K18256', 'K18275', 'K18276', 'K18277', 'K18293', 'K18312', 'K18333', 'K18334', 'K18335', 'K18336', 'K18337', 'K18338', 'K18339', 'K18364', 'K18365', 'K18366', 'K18367', 'K18425', 'K18472', 'K18541', 'K18556', 'K18558', 'K18593', 'K18594', 'K18602', 'K18603', 'K18604', 'K18605', 'K18607', 'K18608', 'K18609', 'K18610', 'K18611', 'K18612', 'K18613', 'K18614', 'K18687', 'K18688', 'K18857', 'K18859', 'K18860', 'K18861', 'K18881', 'K18978', 'K19066', 'K19067', 'K19185', 'K19186', 'K19187', 'K19188', 'K19189', 'K19190', 'K19191', 'K19243', 'K19266', 'K19268', 'K19280', 'K19281', 'K19282', 'K19312', 'K19515', 'K19516', 'K19551', 'K19629', 'K19630', 'K19634', 'K19647', 'K19649', 'K19653', 'K19670', 'K19700', 'K19709', 'K19713', 'K19743', 'K19745', 'K19818', 'K19819', 'K19820', 'K19826', 'K19837', 'K19890', 'K19958', 'K19963', 'K19964', 'K20034', 'K20035', 'K20036', 'K20143', 'K20155', 'K20158', 'K20169', 'K20170', 'K20171', 'K20172', 'K20199', 'K20200', 'K20218', 'K20445', 'K20446', 'K20447', 'K20448', 'K20449', 'K20450', 'K20451', 'K20452', 'K20453', 'K20454', 'K20455', 'K20458', 'K20509', 'K20548', 'K20549', 'K20625', 'K20626', 'K20627', 'K20712', 'K20806', 'K20807', 'K20808', 'K20809', 'K20866', 'K20932', 'K20933', 'K20934', 'K20935', 'K20941', 'K20942', 'K20943', 'K20944', 'K20990', 'K21071', 'K21104', 'K21105', 'K21307', 'K21308', 'K21309', 'K21310', 'K21317', 'K21318', 'K21319', 'K21320', 'K21321', 'K21322', 'K21569', 'K21570', 'K21593', 'K21607', 'K21610', 'K21611', 'K21612', 'K21619', 'K21647', 'K21648', 'K21673', 'K21674', 'K21675', 'K21682', 'K21683', 'K21722', 'K21723', 'K21724', 'K21725', 'K21726', 'K21727', 'K21730', 'K21731', 'K21759', 'K21802', 'K21840', 'K21883', 'K22011', 'K22012', 'K22015', 'K22081', 'K22082', 'K22083', 'K22084', 'K22085', 'K22086', 'K22087', 'K22211', 'K22212', 'K22224', 'K22229', 'K22230', 'K22270', 'K22305', 'K22353', 'K22354', 'K22355', 'K22356', 'K22357', 'K22358', 'K22359', 'K22360', 'K22361', 'K22362', 'K22363', 'K22470', 'K22473', 'K22474', 'K22480', 'K22481', 'K22482', 'K22515', 'K22516', 'K22539', 'K22553', 'K22568', 'K22601', 'K22602', 'K22622', 'K22676', 'K22677', 'K22678', 'K22715', 'K22716', 'K22717', 'K22818', 'K22819', 'K22820', 'K22821', 'K22822', 'K22879', 'K22933', 'K22958', 'K22959', 'K22960', 'K22966', 'K22969', 'K22994', 'K23020', 'K23146', 'K23304', 'K23351', 'K23352', 'K23359', 'K23492', 'K23526', 'K23527', 'K23528', 'K23529', 'K23548', 'K23549', 'K23995', 'K23998', 'K24012', 'K24042', 'K24182', 'K24244', 'K24245', 'K24293', 'K24393', 'K24666', 'K24998', 'K25004', 'K25007', 'K25008', 'K25009', 'K25026', 'K25031', 'K25123', 'K25316', 'K25317', 'K25621', 'K25622', 'K25623', 'K25774', 'K25801', 'K25916', 'K25932', 'K25933', 'K25934', 'K25935', 'K25936', 'K25937', 'K25938', 'K25986', 'K25995', 'K25996', 'K26061', 'K26062', 'K26063', 'K26064', 'K26065', 'K26138', 'K26139', 'K26209', 'K26318', 'K26398', 'K26911', 'K26933', 'K27094', 'K27095', 'K27148', 'K27187', 'K27188', 'K27190', 'K27191', 'K27196', 'K27475', 'K27499', 'K27500', 'K27540', 'K27672', 'K27673', 'K27674', 'K27797', 'K27802', 'K27848', 'K27877', 'K27925', 'K28060', 'K28061', 'K28064', 'K28065', 'K28066', 'K28067', 'K28068', 'K28069', 'K28070', 'K28071', 'K28072', 'K28073', 'K28178', 'K28179', 'K28180', 'K28205', 'K28218', 'K28219', 'K28220', 'K28221', 'K28222', 'K28224', 'K28225', 'K28226', 'K28227', 'K28228', 'K28229', 'K28230', 'K28231', 'K28232', 'K28462', 'K28472', 'K28482', 'K28483', 'K28484', 'K28504', 'K28519', 'K28520', 'K28522', 'K28556', 'K28615', 'K28660', 'K28690', 'K28691', 'K28692', 'K28693', 'K28715', 'K28726', 'K28727', 'K28728', 'K28729', 'K28748', 'K28749', 'K28789', 'K28794', 'K28795', 'K28849', 'K28886', 'K28887', 'K28979'],
    # Part A | negative_control | ko02020
    'two_component': ['K00027', 'K00066', 'K00244', 'K00245', 'K00246', 'K00247', 'K00370', 'K00371', 'K00373', 'K00374', 'K00404', 'K00405', 'K00406', 'K00407', 'K00410', 'K00411', 'K00412', 'K00413', 'K00424', 'K00425', 'K00426', 'K00494', 'K00575', 'K00626', 'K00689', 'K00692', 'K00990', 'K01034', 'K01035', 'K01051', 'K01077', 'K01113', 'K01179', 'K01425', 'K01467', 'K01545', 'K01546', 'K01547', 'K01548', 'K01643', 'K01644', 'K01646', 'K01791', 'K01910', 'K01915', 'K01991', 'K02040', 'K02106', 'K02252', 'K02253', 'K02259', 'K02279', 'K02280', 'K02282', 'K02283', 'K02313', 'K02398', 'K02402', 'K02403', 'K02405', 'K02406', 'K02472', 'K02488', 'K02489', 'K02490', 'K02491', 'K02556', 'K02584', 'K02650', 'K02651', 'K02657', 'K02658', 'K02659', 'K02660', 'K02661', 'K02667', 'K02668', 'K03092', 'K03367', 'K03400', 'K03406', 'K03407', 'K03408', 'K03412', 'K03413', 'K03415', 'K03532', 'K03533', 'K03563', 'K03620', 'K03739', 'K03740', 'K03776', 'K04751', 'K04771', 'K05338', 'K05339', 'K05597', 'K05874', 'K05875', 'K05876', 'K05877', 'K05964', 'K05966', 'K06046', 'K06080', 'K06281', 'K06282', 'K06347', 'K06375', 'K06596', 'K06597', 'K06598', 'K07165', 'K07260', 'K07346', 'K07347', 'K07636', 'K07637', 'K07638', 'K07639', 'K07640', 'K07641', 'K07642', 'K07643', 'K07644', 'K07645', 'K07646', 'K07647', 'K07648', 'K07649', 'K07650', 'K07651', 'K07652', 'K07653', 'K07654', 'K07655', 'K07656', 'K07657', 'K07658', 'K07659', 'K07660', 'K07661', 'K07662', 'K07663', 'K07664', 'K07665', 'K07666', 'K07667', 'K07668', 'K07669', 'K07670', 'K07671', 'K07672', 'K07673', 'K07674', 'K07675', 'K07676', 'K07677', 'K07678', 'K07679', 'K07680', 'K07681', 'K07682', 'K07683', 'K07684', 'K07685', 'K07686', 'K07687', 'K07688', 'K07689', 'K07690', 'K07691', 'K07692', 'K07693', 'K07694', 'K07695', 'K07696', 'K07697', 'K07698', 'K07699', 'K07700', 'K07701', 'K07702', 'K07703', 'K07704', 'K07705', 'K07706', 'K07707', 'K07708', 'K07709', 'K07710', 'K07711', 'K07712', 'K07713', 'K07714', 'K07715', 'K07716', 'K07717', 'K07718', 'K07719', 'K07720', 'K07768', 'K07769', 'K07770', 'K07771', 'K07772', 'K07773', 'K07774', 'K07775', 'K07776', 'K07777', 'K07778', 'K07780', 'K07781', 'K07782', 'K07783', 'K07784', 'K07785', 'K07786', 'K07787', 'K07788', 'K07789', 'K07790', 'K07792', 'K07793', 'K07794', 'K07795', 'K07796', 'K07797', 'K07798', 'K07799', 'K07800', 'K07801', 'K07803', 'K07804', 'K07805', 'K07806', 'K07810', 'K07811', 'K07813', 'K08082', 'K08083', 'K08177', 'K08348', 'K08349', 'K08350', 'K08357', 'K08358', 'K08359', 'K08372', 'K08475', 'K08476', 'K08477', 'K08478', 'K08479', 'K08641', 'K08738', 'K08926', 'K08927', 'K08928', 'K08929', 'K08930', 'K08939', 'K09474', 'K09475', 'K09476', 'K09477', 'K09696', 'K09697', 'K10001', 'K10002', 'K10003', 'K10004', 'K10125', 'K10126', 'K10255', 'K10681', 'K10682', 'K10697', 'K10715', 'K10850', 'K10851', 'K10909', 'K10910', 'K10911', 'K10912', 'K10913', 'K10914', 'K10916', 'K10941', 'K10942', 'K10943', 'K11103', 'K11230', 'K11231', 'K11232', 'K11233', 'K11326', 'K11327', 'K11328', 'K11329', 'K11330', 'K11331', 'K11332', 'K11354', 'K11355', 'K11356', 'K11357', 'K11382', 'K11383', 'K11384', 'K11443', 'K11444', 'K11520', 'K11521', 'K11522', 'K11523', 'K11524', 'K11525', 'K11526', 'K11601', 'K11602', 'K11603', 'K11614', 'K11615', 'K11616', 'K11617', 'K11618', 'K11619', 'K11620', 'K11621', 'K11622', 'K11623', 'K11624', 'K11625', 'K11626', 'K11629', 'K11630', 'K11631', 'K11632', 'K11633', 'K11634', 'K11635', 'K11636', 'K11637', 'K11638', 'K11639', 'K11640', 'K11641', 'K11688', 'K11689', 'K11690', 'K11691', 'K11692', 'K11711', 'K11712', 'K12292', 'K12293', 'K12294', 'K12295', 'K12296', 'K12340', 'K12415', 'K12510', 'K12511', 'K12530', 'K12531', 'K12532', 'K13040', 'K13041', 'K13061', 'K13486', 'K13487', 'K13488', 'K13489', 'K13490', 'K13491', 'K13532', 'K13533', 'K13584', 'K13587', 'K13588', 'K13589', 'K13598', 'K13599', 'K13815', 'K13816', 'K13924', 'K13927', 'K13991', 'K13994', 'K14188', 'K14205', 'K14978', 'K14979', 'K14980', 'K14981', 'K14982', 'K14983', 'K14986', 'K14987', 'K14988', 'K14989', 'K15011', 'K15012', 'K15739', 'K15841', 'K15850', 'K15851', 'K15853', 'K15854', 'K15859', 'K15860', 'K15861', 'K15862', 'K16692', 'K16712', 'K16713', 'K17060', 'K17061', 'K18072', 'K18073', 'K18093', 'K18094', 'K18095', 'K18321', 'K18322', 'K18323', 'K18324', 'K18326', 'K18344', 'K18345', 'K18346', 'K18347', 'K18348', 'K18349', 'K18350', 'K18351', 'K18352', 'K18353', 'K18354', 'K18444', 'K18856', 'K18866', 'K18940', 'K18941', 'K18986', 'K18987', 'K19077', 'K19078', 'K19079', 'K19080', 'K19081', 'K19082', 'K19083', 'K19084', 'K19609', 'K19610', 'K19611', 'K19615', 'K19616', 'K19617', 'K19618', 'K19620', 'K19621', 'K19622', 'K19624', 'K19641', 'K19661', 'K19666', 'K19667', 'K19668', 'K19690', 'K19691', 'K19692', 'K20263', 'K20264', 'K20339', 'K20340', 'K20482', 'K20483', 'K20484', 'K20485', 'K20486', 'K20487', 'K20488', 'K20489', 'K20490', 'K20491', 'K20492', 'K20494', 'K20552', 'K20973', 'K20974', 'K20975', 'K20976', 'K20977', 'K20978', 'K21393', 'K22501', 'K23236', 'K23514', 'K23548', 'K23549', 'K23676', 'K25211', 'K25212', 'K25213', 'K25307', 'K25864', 'K27076', 'K27077', 'K27078', 'K27079', 'K27080', 'K27105', 'K27882', 'K27883', 'K27884', 'K27885'],
    # Part A | negative_control | ko02010−metal
    'abc_transporters': ['K01995', 'K01996', 'K01997', 'K01998', 'K01999', 'K02000', 'K02001', 'K02002', 'K02011', 'K02017', 'K02018', 'K02020', 'K02036', 'K02037', 'K02038', 'K02040', 'K02041', 'K02042', 'K02044', 'K02045', 'K02046', 'K02047', 'K02048', 'K02062', 'K02063', 'K02064', 'K02065', 'K02066', 'K02067', 'K02071', 'K02072', 'K02073', 'K02193', 'K02194', 'K02195', 'K02196', 'K02424', 'K02471', 'K05031', 'K05032', 'K05033', 'K05641', 'K05642', 'K05643', 'K05644', 'K05645', 'K05646', 'K05647', 'K05648', 'K05649', 'K05650', 'K05651', 'K05652', 'K05653', 'K05654', 'K05655', 'K05656', 'K05657', 'K05658', 'K05659', 'K05660', 'K05661', 'K05662', 'K05664', 'K05665', 'K05666', 'K05667', 'K05668', 'K05669', 'K05670', 'K05671', 'K05672', 'K05673', 'K05674', 'K05675', 'K05676', 'K05677', 'K05678', 'K05679', 'K05680', 'K05681', 'K05682', 'K05683', 'K05684', 'K05685', 'K05772', 'K05773', 'K05776', 'K05813', 'K05814', 'K05815', 'K05816', 'K05845', 'K05846', 'K05847', 'K06073', 'K06074', 'K06159', 'K06160', 'K06161', 'K06726', 'K06857', 'K06858', 'K06861', 'K07091', 'K07122', 'K07323', 'K07335', 'K08711', 'K08712', 'K09688', 'K09689', 'K09690', 'K09691', 'K09692', 'K09693', 'K09694', 'K09695', 'K09696', 'K09697', 'K09808', 'K09810', 'K09811', 'K09812', 'K09813', 'K09814', 'K09969', 'K09970', 'K09971', 'K09972', 'K09996', 'K09997', 'K09998', 'K09999', 'K10000', 'K10001', 'K10002', 'K10003', 'K10004', 'K10005', 'K10006', 'K10007', 'K10008', 'K10009', 'K10010', 'K10013', 'K10014', 'K10015', 'K10016', 'K10017', 'K10018', 'K10019', 'K10020', 'K10021', 'K10022', 'K10023', 'K10024', 'K10025', 'K10036', 'K10037', 'K10038', 'K10039', 'K10040', 'K10041', 'K10107', 'K10108', 'K10109', 'K10110', 'K10111', 'K10112', 'K10117', 'K10118', 'K10119', 'K10188', 'K10189', 'K10190', 'K10191', 'K10192', 'K10193', 'K10194', 'K10195', 'K10196', 'K10197', 'K10198', 'K10199', 'K10200', 'K10201', 'K10202', 'K10227', 'K10228', 'K10229', 'K10232', 'K10233', 'K10234', 'K10235', 'K10236', 'K10237', 'K10238', 'K10240', 'K10241', 'K10242', 'K10439', 'K10440', 'K10441', 'K10537', 'K10538', 'K10539', 'K10540', 'K10541', 'K10542', 'K10543', 'K10544', 'K10545', 'K10546', 'K10547', 'K10548', 'K10549', 'K10550', 'K10551', 'K10552', 'K10553', 'K10554', 'K10555', 'K10556', 'K10557', 'K10558', 'K10559', 'K10560', 'K10561', 'K10562', 'K10820', 'K10823', 'K10824', 'K10829', 'K10831', 'K11004', 'K11050', 'K11051', 'K11069', 'K11070', 'K11071', 'K11072', 'K11073', 'K11074', 'K11075', 'K11076', 'K11077', 'K11078', 'K11079', 'K11080', 'K11081', 'K11082', 'K11083', 'K11084', 'K11085', 'K11631', 'K11632', 'K11720', 'K11950', 'K11951', 'K11952', 'K11953', 'K11954', 'K11955', 'K11956', 'K11957', 'K11958', 'K11959', 'K11960', 'K11961', 'K11962', 'K11963', 'K12292', 'K12368', 'K12369', 'K12370', 'K12371', 'K12372', 'K12536', 'K12541', 'K13409', 'K13889', 'K13890', 'K13891', 'K13892', 'K13893', 'K13894', 'K13895', 'K13896', 'K14698', 'K14699', 'K15495', 'K15496', 'K15497', 'K15551', 'K15552', 'K15553', 'K15554', 'K15555', 'K15556', 'K15557', 'K15558', 'K15576', 'K15577', 'K15578', 'K15579', 'K15580', 'K15581', 'K15582', 'K15583', 'K15584', 'K15585', 'K15586', 'K15587', 'K15598', 'K15599', 'K15600', 'K15628', 'K15770', 'K15771', 'K15772', 'K16012', 'K16013', 'K16014', 'K16199', 'K16200', 'K16201', 'K16202', 'K16785', 'K16786', 'K16787', 'K16905', 'K16906', 'K16907', 'K16916', 'K16917', 'K16918', 'K16919', 'K16920', 'K16921', 'K16956', 'K16957', 'K16958', 'K16959', 'K16960', 'K16961', 'K16962', 'K16963', 'K17062', 'K17063', 'K17073', 'K17074', 'K17076', 'K17077', 'K17202', 'K17203', 'K17204', 'K17205', 'K17206', 'K17207', 'K17208', 'K17209', 'K17210', 'K17213', 'K17214', 'K17215', 'K17234', 'K17235', 'K17236', 'K17237', 'K17238', 'K17239', 'K17240', 'K17241', 'K17242', 'K17243', 'K17244', 'K17245', 'K17246', 'K17311', 'K17312', 'K17313', 'K17314', 'K17315', 'K17316', 'K17317', 'K17318', 'K17319', 'K17320', 'K17321', 'K17322', 'K17323', 'K17324', 'K17325', 'K17326', 'K17327', 'K17328', 'K17329', 'K17330', 'K17331', 'K18104', 'K18216', 'K18217', 'K18230', 'K18231', 'K18232', 'K18233', 'K18887', 'K18888', 'K18889', 'K18890', 'K18891', 'K18892', 'K18893', 'K18894', 'K18895', 'K19079', 'K19080', 'K19083', 'K19084', 'K19226', 'K19227', 'K19228', 'K19229', 'K19230', 'K19309', 'K19310', 'K19340', 'K19341', 'K19349', 'K19350', 'K20344', 'K20386', 'K20459', 'K20460', 'K20461', 'K20490', 'K20491', 'K20492', 'K20494', 'K22921', 'K22922', 'K22923', 'K23055', 'K23056', 'K23057', 'K23058', 'K23059', 'K23060', 'K23061', 'K23062', 'K23063', 'K23064', 'K23125', 'K23163', 'K23181', 'K23182', 'K23183', 'K23184', 'K23227', 'K23228', 'K23508', 'K23509', 'K23510', 'K23511', 'K23512', 'K23513', 'K23535', 'K23536', 'K23537', 'K23545', 'K23546', 'K23547', 'K24821', 'K25819'],
    # Part A | negative_control | ko02024
    'quorum_sensing': ['K00494', 'K01114', 'K01218', 'K01318', 'K01364', 'K01399', 'K01497', 'K01580', 'K01626', 'K01635', 'K01657', 'K01658', 'K01728', 'K01897', 'K01995', 'K01996', 'K01997', 'K01998', 'K01999', 'K02031', 'K02032', 'K02033', 'K02034', 'K02035', 'K02052', 'K02053', 'K02054', 'K02055', 'K02250', 'K02251', 'K02252', 'K02253', 'K02402', 'K02403', 'K02490', 'K03070', 'K03071', 'K03073', 'K03075', 'K03076', 'K03106', 'K03110', 'K03210', 'K03217', 'K03400', 'K03666', 'K06046', 'K06352', 'K06353', 'K06354', 'K06355', 'K06356', 'K06358', 'K06359', 'K06360', 'K06361', 'K06363', 'K06364', 'K06365', 'K06366', 'K06369', 'K06375', 'K06998', 'K07173', 'K07344', 'K07645', 'K07666', 'K07667', 'K07680', 'K07691', 'K07692', 'K07699', 'K07706', 'K07707', 'K07711', 'K07715', 'K07781', 'K07782', 'K07800', 'K07813', 'K08321', 'K08605', 'K08642', 'K08777', 'K09823', 'K09936', 'K10555', 'K10556', 'K10557', 'K10558', 'K10715', 'K10823', 'K10909', 'K10910', 'K10911', 'K10912', 'K10913', 'K10914', 'K10915', 'K10916', 'K10917', 'K11006', 'K11007', 'K11031', 'K11033', 'K11034', 'K11035', 'K11036', 'K11037', 'K11039', 'K11063', 'K11216', 'K11530', 'K11531', 'K11752', 'K12257', 'K12292', 'K12293', 'K12294', 'K12295', 'K12296', 'K12415', 'K12789', 'K12990', 'K13060', 'K13061', 'K13062', 'K13063', 'K13075', 'K13815', 'K13816', 'K14051', 'K14645', 'K14982', 'K14983', 'K15580', 'K15581', 'K15582', 'K15583', 'K15654', 'K15655', 'K15656', 'K15657', 'K15850', 'K15851', 'K15852', 'K15853', 'K15854', 'K16619', 'K17940', 'K18000', 'K18001', 'K18002', 'K18003', 'K18096', 'K18098', 'K18099', 'K18100', 'K18101', 'K18139', 'K18304', 'K18306', 'K18307', 'K18315', 'K18316', 'K18317', 'K18318', 'K18319', 'K19666', 'K19731', 'K19732', 'K19733', 'K19734', 'K19735', 'K20086', 'K20087', 'K20088', 'K20089', 'K20090', 'K20248', 'K20249', 'K20250', 'K20252', 'K20253', 'K20256', 'K20257', 'K20258', 'K20259', 'K20260', 'K20261', 'K20262', 'K20263', 'K20264', 'K20265', 'K20266', 'K20267', 'K20268', 'K20269', 'K20270', 'K20271', 'K20272', 'K20273', 'K20274', 'K20275', 'K20276', 'K20277', 'K20321', 'K20322', 'K20323', 'K20324', 'K20325', 'K20326', 'K20327', 'K20328', 'K20329', 'K20330', 'K20331', 'K20332', 'K20333', 'K20334', 'K20335', 'K20336', 'K20337', 'K20338', 'K20339', 'K20340', 'K20341', 'K20342', 'K20343', 'K20344', 'K20345', 'K20373', 'K20374', 'K20375', 'K20376', 'K20377', 'K20378', 'K20379', 'K20380', 'K20381', 'K20382', 'K20383', 'K20384', 'K20385', 'K20386', 'K20387', 'K20388', 'K20389', 'K20390', 'K20391', 'K20480', 'K20481', 'K20482', 'K20483', 'K20484', 'K20485', 'K20486', 'K20487', 'K20488', 'K20489', 'K20490', 'K20491', 'K20492', 'K20494', 'K20527', 'K20528', 'K20529', 'K20530', 'K20531', 'K20532', 'K20533', 'K20539', 'K20540', 'K20552', 'K20554', 'K20555', 'K22954', 'K22955', 'K22956', 'K22957', 'K22968', 'K23133', 'K25873'],
    # Part B | core_metabolism | ko01200
    'carbohydrate_metab': ['K00018', 'K00023', 'K00024', 'K00025', 'K00026', 'K00027', 'K00028', 'K00029', 'K00030', 'K00031', 'K00033', 'K00034', 'K00036', 'K00043', 'K00051', 'K00058', 'K00074', 'K00116', 'K00121', 'K00122', 'K00123', 'K00124', 'K00125', 'K00126', 'K00127', 'K00131', 'K00134', 'K00140', 'K00148', 'K00150', 'K00161', 'K00162', 'K00163', 'K00164', 'K00169', 'K00170', 'K00171', 'K00172', 'K00174', 'K00175', 'K00176', 'K00177', 'K00189', 'K00192', 'K00193', 'K00194', 'K00195', 'K00196', 'K00197', 'K00198', 'K00200', 'K00201', 'K00202', 'K00203', 'K00204', 'K00205', 'K00209', 'K00232', 'K00234', 'K00235', 'K00236', 'K00237', 'K00239', 'K00240', 'K00241', 'K00242', 'K00244', 'K00245', 'K00246', 'K00247', 'K00248', 'K00261', 'K00281', 'K00282', 'K00283', 'K00297', 'K00317', 'K00319', 'K00320', 'K00382', 'K00399', 'K00401', 'K00402', 'K00577', 'K00578', 'K00579', 'K00580', 'K00581', 'K00582', 'K00583', 'K00584', 'K00600', 'K00605', 'K00615', 'K00616', 'K00625', 'K00626', 'K00627', 'K00640', 'K00658', 'K00672', 'K00814', 'K00827', 'K00830', 'K00831', 'K00844', 'K00845', 'K00850', 'K00851', 'K00855', 'K00863', 'K00873', 'K00874', 'K00886', 'K00918', 'K00925', 'K00926', 'K00927', 'K00948', 'K01006', 'K01007', 'K01053', 'K01057', 'K01070', 'K01079', 'K01086', 'K01100', 'K01455', 'K01491', 'K01499', 'K01500', 'K01595', 'K01601', 'K01602', 'K01610', 'K01616', 'K01622', 'K01623', 'K01624', 'K01625', 'K01637', 'K01638', 'K01647', 'K01659', 'K01676', 'K01677', 'K01678', 'K01679', 'K01681', 'K01682', 'K01689', 'K01690', 'K01715', 'K01720', 'K01738', 'K01752', 'K01754', 'K01782', 'K01783', 'K01803', 'K01807', 'K01808', 'K01810', 'K01825', 'K01834', 'K01846', 'K01847', 'K01848', 'K01849', 'K01895', 'K01899', 'K01900', 'K01902', 'K01903', 'K01913', 'K01938', 'K01948', 'K01958', 'K01959', 'K01960', 'K01961', 'K01962', 'K01963', 'K01964', 'K01965', 'K01966', 'K02160', 'K02203', 'K02437', 'K02446', 'K03388', 'K03389', 'K03390', 'K03396', 'K03417', 'K03737', 'K03738', 'K03781', 'K03841', 'K04480', 'K04835', 'K05298', 'K05299', 'K05308', 'K05605', 'K05606', 'K05942', 'K06859', 'K07404', 'K07511', 'K07516', 'K08074', 'K08093', 'K08094', 'K08264', 'K08265', 'K08318', 'K08691', 'K08692', 'K09709', 'K09788', 'K10713', 'K10714', 'K10944', 'K10945', 'K10946', 'K11260', 'K11261', 'K11263', 'K11389', 'K11395', 'K11517', 'K11529', 'K11532', 'K11645', 'K12406', 'K12407', 'K13034', 'K13788', 'K13810', 'K13812', 'K13831', 'K13937', 'K13942', 'K14028', 'K14029', 'K14067', 'K14080', 'K14081', 'K14082', 'K14083', 'K14084', 'K14126', 'K14127', 'K14128', 'K14138', 'K14272', 'K14446', 'K14447', 'K14448', 'K14449', 'K14451', 'K14454', 'K14455', 'K14465', 'K14466', 'K14467', 'K14468', 'K14469', 'K14470', 'K14471', 'K14472', 'K14534', 'K14729', 'K15016', 'K15017', 'K15018', 'K15019', 'K15020', 'K15022', 'K15023', 'K15024', 'K15036', 'K15037', 'K15038', 'K15039', 'K15052', 'K15230', 'K15231', 'K15232', 'K15233', 'K15234', 'K15633', 'K15634', 'K15635', 'K15893', 'K15916', 'K15918', 'K15919', 'K16157', 'K16158', 'K16159', 'K16160', 'K16161', 'K16162', 'K16176', 'K16177', 'K16178', 'K16179', 'K16305', 'K16306', 'K16370', 'K17066', 'K17067', 'K17069', 'K17100', 'K17829', 'K17865', 'K17989', 'K18020', 'K18021', 'K18022', 'K18118', 'K18119', 'K18120', 'K18121', 'K18122', 'K18124', 'K18125', 'K18126', 'K18127', 'K18128', 'K18209', 'K18210', 'K18472', 'K18556', 'K18557', 'K18558', 'K18559', 'K18560', 'K18859', 'K18860', 'K18861', 'K18978', 'K19243', 'K19268', 'K19269', 'K19280', 'K19281', 'K19282', 'K19312', 'K20455', 'K21071', 'K21682', 'K22015', 'K22305', 'K22480', 'K22481', 'K22482', 'K22515', 'K22516', 'K22568', 'K22969', 'K23146', 'K23304', 'K24182', 'K25007', 'K25008', 'K25026', 'K25031', 'K25123', 'K25124', 'K25528', 'K25774', 'K25801', 'K27094', 'K27095', 'K27394', 'K27802', 'K28462', 'K28726', 'K28727', 'K28728', 'K28729', 'K28761', 'K28886', 'K28887'],
    # Part B | core_metabolism | ko00190
    'energy_metab': ['K00233', 'K00234', 'K00235', 'K00236', 'K00237', 'K00239', 'K00240', 'K00241', 'K00242', 'K00244', 'K00245', 'K00246', 'K00247', 'K00330', 'K00331', 'K00332', 'K00333', 'K00334', 'K00335', 'K00336', 'K00337', 'K00338', 'K00339', 'K00340', 'K00341', 'K00342', 'K00343', 'K00404', 'K00405', 'K00406', 'K00407', 'K00410', 'K00411', 'K00412', 'K00413', 'K00414', 'K00415', 'K00416', 'K00417', 'K00418', 'K00419', 'K00420', 'K00424', 'K00425', 'K00426', 'K00937', 'K01507', 'K01535', 'K01542', 'K01543', 'K01544', 'K01549', 'K02107', 'K02108', 'K02109', 'K02110', 'K02111', 'K02112', 'K02113', 'K02114', 'K02115', 'K02117', 'K02118', 'K02119', 'K02120', 'K02121', 'K02122', 'K02123', 'K02124', 'K02125', 'K02126', 'K02127', 'K02128', 'K02129', 'K02130', 'K02131', 'K02132', 'K02133', 'K02134', 'K02135', 'K02136', 'K02137', 'K02138', 'K02139', 'K02140', 'K02141', 'K02142', 'K02143', 'K02144', 'K02145', 'K02146', 'K02147', 'K02148', 'K02149', 'K02150', 'K02151', 'K02152', 'K02153', 'K02154', 'K02155', 'K02256', 'K02257', 'K02258', 'K02259', 'K02260', 'K02261', 'K02262', 'K02263', 'K02264', 'K02265', 'K02266', 'K02267', 'K02268', 'K02269', 'K02270', 'K02271', 'K02272', 'K02273', 'K02274', 'K02275', 'K02276', 'K02277', 'K02297', 'K02298', 'K02299', 'K02300', 'K02826', 'K02827', 'K02828', 'K02829', 'K03661', 'K03662', 'K03878', 'K03879', 'K03880', 'K03881', 'K03882', 'K03883', 'K03884', 'K03885', 'K03886', 'K03887', 'K03888', 'K03889', 'K03890', 'K03891', 'K03934', 'K03935', 'K03936', 'K03937', 'K03938', 'K03939', 'K03940', 'K03941', 'K03942', 'K03943', 'K03944', 'K03945', 'K03946', 'K03947', 'K03948', 'K03949', 'K03950', 'K03951', 'K03952', 'K03953', 'K03954', 'K03955', 'K03956', 'K03957', 'K03958', 'K03959', 'K03960', 'K03961', 'K03962', 'K03963', 'K03964', 'K03965', 'K03966', 'K03967', 'K03968', 'K05572', 'K05573', 'K05574', 'K05575', 'K05576', 'K05577', 'K05578', 'K05579', 'K05580', 'K05581', 'K05582', 'K05583', 'K05584', 'K05585', 'K05586', 'K05587', 'K05588', 'K06019', 'K08738', 'K11351', 'K11352', 'K11353', 'K11725', 'K11726', 'K13378', 'K13380', 'K15408', 'K15862', 'K15863', 'K15986', 'K18859', 'K18860', 'K22468', 'K22501', 'K24007', 'K24008', 'K24009', 'K24010', 'K24011', 'K25801', 'K25995', 'K25996', 'K27109'],
    # Part B | core_metabolism | ko01212
    'lipid_metab': ['K00022', 'K00059', 'K00208', 'K00209', 'K00232', 'K00248', 'K00249', 'K00255', 'K00507', 'K00626', 'K00632', 'K00645', 'K00647', 'K00648', 'K00665', 'K00667', 'K00668', 'K01074', 'K01692', 'K01716', 'K01782', 'K01825', 'K01897', 'K01961', 'K01962', 'K01963', 'K02160', 'K02371', 'K02372', 'K03921', 'K03922', 'K06445', 'K07508', 'K07509', 'K07511', 'K07512', 'K07513', 'K07514', 'K07515', 'K07516', 'K08764', 'K08765', 'K08766', 'K09458', 'K09478', 'K09479', 'K10203', 'K10205', 'K10224', 'K10226', 'K10244', 'K10245', 'K10246', 'K10247', 'K10248', 'K10249', 'K10250', 'K10251', 'K10256', 'K10258', 'K10527', 'K10703', 'K10780', 'K10781', 'K11262', 'K11263', 'K11533', 'K11539', 'K12405', 'K13370', 'K13767', 'K15013', 'K16363', 'K18472', 'K18473', 'K18660', 'K19523', 'K19524', 'K22540', 'K22541', 'K22769', 'K22770', 'K22993', 'K23006'],
    # Part B | core_metabolism | ko01232
    'nucleotide_metab': ['K00087', 'K00088', 'K00106', 'K00364', 'K00524', 'K00525', 'K00526', 'K00527', 'K00560', 'K00756', 'K00757', 'K00758', 'K00759', 'K00760', 'K00761', 'K00769', 'K00856', 'K00857', 'K00876', 'K00892', 'K00893', 'K00904', 'K00939', 'K00940', 'K00942', 'K00943', 'K00944', 'K00945', 'K01081', 'K01129', 'K01239', 'K01240', 'K01241', 'K01250', 'K01485', 'K01486', 'K01487', 'K01488', 'K01489', 'K01490', 'K01493', 'K01494', 'K01509', 'K01510', 'K01511', 'K01513', 'K01519', 'K01520', 'K01529', 'K01756', 'K01937', 'K01939', 'K01951', 'K02566', 'K02825', 'K03365', 'K03465', 'K03783', 'K03784', 'K03787', 'K03816', 'K04765', 'K05810', 'K05961', 'K06287', 'K06928', 'K06952', 'K06966', 'K07023', 'K07043', 'K08320', 'K08693', 'K08722', 'K08723', 'K09887', 'K09903', 'K09913', 'K10213', 'K10353', 'K10807', 'K10808', 'K11177', 'K11178', 'K11751', 'K12304', 'K12305', 'K12700', 'K13479', 'K13480', 'K13481', 'K13482', 'K13483', 'K13799', 'K13800', 'K13809', 'K13998', 'K14641', 'K14642', 'K15518', 'K15519', 'K15780', 'K16855', 'K16904', 'K18532', 'K18533', 'K18550', 'K18931', 'K19572', 'K19836', 'K19970', 'K20881', 'K21053', 'K21057', 'K21636', 'K22026', 'K24242', 'K25434', 'K25435', 'K25589', 'K25641', 'K26110', 'K26211', 'K26212'],
    # Part B | core_metabolism | ko01230
    'aa_metab': ['K00003', 'K00013', 'K00014', 'K00030', 'K00031', 'K00052', 'K00053', 'K00058', 'K00133', 'K00134', 'K00143', 'K00145', 'K00147', 'K00150', 'K00211', 'K00215', 'K00220', 'K00264', 'K00265', 'K00266', 'K00286', 'K00290', 'K00293', 'K00500', 'K00548', 'K00549', 'K00600', 'K00611', 'K00615', 'K00616', 'K00618', 'K00619', 'K00620', 'K00640', 'K00641', 'K00651', 'K00674', 'K00765', 'K00766', 'K00789', 'K00800', 'K00811', 'K00812', 'K00813', 'K00814', 'K00817', 'K00818', 'K00821', 'K00826', 'K00831', 'K00832', 'K00836', 'K00838', 'K00841', 'K00850', 'K00872', 'K00873', 'K00891', 'K00918', 'K00927', 'K00928', 'K00930', 'K00931', 'K00948', 'K01079', 'K01089', 'K01243', 'K01438', 'K01439', 'K01476', 'K01496', 'K01523', 'K01586', 'K01609', 'K01620', 'K01622', 'K01623', 'K01624', 'K01626', 'K01647', 'K01649', 'K01652', 'K01653', 'K01655', 'K01656', 'K01657', 'K01658', 'K01659', 'K01663', 'K01681', 'K01682', 'K01687', 'K01689', 'K01693', 'K01694', 'K01695', 'K01696', 'K01697', 'K01702', 'K01703', 'K01704', 'K01705', 'K01713', 'K01714', 'K01733', 'K01735', 'K01736', 'K01738', 'K01739', 'K01750', 'K01752', 'K01754', 'K01755', 'K01758', 'K01760', 'K01778', 'K01783', 'K01803', 'K01807', 'K01808', 'K01814', 'K01817', 'K01834', 'K01850', 'K01914', 'K01915', 'K01940', 'K01948', 'K01953', 'K01958', 'K01959', 'K01960', 'K02203', 'K02204', 'K02500', 'K02501', 'K02502', 'K03340', 'K03785', 'K03786', 'K03856', 'K04092', 'K04093', 'K04486', 'K04516', 'K04517', 'K04518', 'K05359', 'K05602', 'K05822', 'K05823', 'K05824', 'K05827', 'K05828', 'K05829', 'K05830', 'K05831', 'K05942', 'K06001', 'K06208', 'K06209', 'K07173', 'K08093', 'K08094', 'K09011', 'K09065', 'K09758', 'K10150', 'K10206', 'K10977', 'K10978', 'K11067', 'K11258', 'K11358', 'K11389', 'K11645', 'K11755', 'K12339', 'K12406', 'K12524', 'K12525', 'K12526', 'K12657', 'K12659', 'K13034', 'K13497', 'K13498', 'K13501', 'K13503', 'K13810', 'K13812', 'K13829', 'K13830', 'K13831', 'K13832', 'K13853', 'K14152', 'K14155', 'K14170', 'K14187', 'K14260', 'K14267', 'K14272', 'K14454', 'K14455', 'K14677', 'K14681', 'K14682', 'K15227', 'K15633', 'K15634', 'K15635', 'K15849', 'K16305', 'K16306', 'K16370', 'K16792', 'K16793', 'K17069', 'K17216', 'K17217', 'K17450', 'K17462', 'K17753', 'K17989', 'K18649', 'K19412', 'K21071', 'K22305', 'K22476', 'K22477', 'K22478', 'K22846', 'K23304', 'K24017', 'K24018', 'K24034', 'K24042', 'K25528', 'K27394', 'K27802', 'K28842'],
    # Part B | core_metabolism | ko00543
    'glycan_biosyn': ['K00640', 'K02851', 'K02852', 'K03208', 'K03606', 'K03818', 'K03819', 'K11936', 'K11937', 'K12582', 'K13656', 'K13657', 'K13658', 'K13659', 'K13660', 'K13663', 'K13664', 'K13665', 'K13683', 'K13684', 'K16555', 'K16556', 'K16557', 'K16558', 'K16560', 'K16562', 'K16563', 'K16564', 'K16566', 'K16568', 'K16700', 'K16701', 'K16702', 'K16703', 'K16707', 'K16710', 'K19290', 'K19291', 'K19293', 'K19294', 'K19295', 'K19296', 'K20921', 'K20922', 'K20997', 'K20999', 'K21001', 'K21002', 'K21154', 'K21461', 'K25205', 'K25875', 'K25886', 'K25887', 'K25888', 'K25889', 'K25890', 'K25891', 'K25892', 'K25902', 'K25903', 'K25904', 'K25905', 'K25906', 'K25907', 'K25908', 'K25909', 'K25910', 'K25911', 'K25912', 'K25913', 'K25957', 'K25958'],
    # Part B | core_metabolism | ko01240
    'cofactor_vitamin': ['K00002', 'K00012', 'K00059', 'K00072', 'K00077', 'K00082', 'K00097', 'K00103', 'K00128', 'K00208', 'K00225', 'K00226', 'K00228', 'K00230', 'K00231', 'K00254', 'K00275', 'K00278', 'K00287', 'K00288', 'K00300', 'K00355', 'K00382', 'K00435', 'K00452', 'K00453', 'K00457', 'K00463', 'K00486', 'K00515', 'K00568', 'K00589', 'K00591', 'K00595', 'K00600', 'K00606', 'K00608', 'K00609', 'K00610', 'K00643', 'K00647', 'K00652', 'K00699', 'K00762', 'K00763', 'K00767', 'K00768', 'K00788', 'K00789', 'K00793', 'K00794', 'K00796', 'K00798', 'K00826', 'K00831', 'K00833', 'K00858', 'K00859', 'K00861', 'K00867', 'K00868', 'K00877', 'K00878', 'K00939', 'K00940', 'K00941', 'K00944', 'K00946', 'K00949', 'K00950', 'K00953', 'K00954', 'K00963', 'K00966', 'K00969', 'K01012', 'K01053', 'K01077', 'K01113', 'K01195', 'K01307', 'K01432', 'K01440', 'K01465', 'K01491', 'K01495', 'K01497', 'K01498', 'K01500', 'K01556', 'K01579', 'K01591', 'K01598', 'K01599', 'K01633', 'K01661', 'K01664', 'K01665', 'K01698', 'K01719', 'K01737', 'K01749', 'K01756', 'K01772', 'K01809', 'K01845', 'K01885', 'K01906', 'K01911', 'K01916', 'K01918', 'K01919', 'K01920', 'K01922', 'K01930', 'K01935', 'K01937', 'K01938', 'K01939', 'K01947', 'K01950', 'K01954', 'K01955', 'K01956', 'K02169', 'K02170', 'K02188', 'K02189', 'K02190', 'K02191', 'K02201', 'K02224', 'K02225', 'K02226', 'K02227', 'K02228', 'K02229', 'K02230', 'K02231', 'K02232', 'K02233', 'K02257', 'K02259', 'K02302', 'K02303', 'K02304', 'K02318', 'K02372', 'K02492', 'K02495', 'K02496', 'K02548', 'K02549', 'K02551', 'K02552', 'K02619', 'K02823', 'K02858', 'K03146', 'K03147', 'K03148', 'K03149', 'K03150', 'K03151', 'K03153', 'K03179', 'K03181', 'K03182', 'K03183', 'K03184', 'K03185', 'K03186', 'K03342', 'K03394', 'K03399', 'K03472', 'K03473', 'K03474', 'K03517', 'K03525', 'K03635', 'K03637', 'K03638', 'K03639', 'K03644', 'K03707', 'K03750', 'K03793', 'K03794', 'K03795', 'K03800', 'K03801', 'K03809', 'K03831', 'K04032', 'K04487', 'K04719', 'K05357', 'K05884', 'K05895', 'K05928', 'K05934', 'K05936', 'K05979', 'K06034', 'K06042', 'K06125', 'K06126', 'K06127', 'K06134', 'K06210', 'K06215', 'K06897', 'K06914', 'K06982', 'K06989', 'K07072', 'K07130', 'K07144', 'K07758', 'K08097', 'K08281', 'K08310', 'K08679', 'K08680', 'K08681', 'K08973', 'K09007', 'K09458', 'K09680', 'K09698', 'K09722', 'K09733', 'K09789', 'K09833', 'K09834', 'K09882', 'K09883', 'K09903', 'K10046', 'K10047', 'K10105', 'K10106', 'K10810', 'K10977', 'K10978', 'K11146', 'K11152', 'K11153', 'K11161', 'K11204', 'K11205', 'K11212', 'K11540', 'K11541', 'K11752', 'K11753', 'K11754', 'K11780', 'K11781', 'K11782', 'K11783', 'K11784', 'K11785', 'K12073', 'K12234', 'K12501', 'K12502', 'K13038', 'K13039', 'K13248', 'K13367', 'K13369', 'K13402', 'K13403', 'K13421', 'K13540', 'K13541', 'K13542', 'K13543', 'K13799', 'K13800', 'K13809', 'K13939', 'K13940', 'K13941', 'K13950', 'K13998', 'K14153', 'K14154', 'K14163', 'K14190', 'K14263', 'K14652', 'K14654', 'K14655', 'K14759', 'K14760', 'K14941', 'K15376', 'K15734', 'K15740', 'K16593', 'K16792', 'K16793', 'K16869', 'K17364', 'K17497', 'K17744', 'K17745', 'K17828', 'K17872', 'K18240', 'K18278', 'K18284', 'K18285', 'K18286', 'K18482', 'K18532', 'K18533', 'K18534', 'K18586', 'K18800', 'K18853', 'K18933', 'K19221', 'K19222', 'K19267', 'K19560', 'K19561', 'K19562', 'K19563', 'K19642', 'K19793', 'K19965', 'K20457', 'K20810', 'K20860', 'K20861', 'K20862', 'K20884', 'K20967', 'K21063', 'K21064', 'K21142', 'K21219', 'K21220', 'K21456', 'K21479', 'K21610', 'K21611', 'K21612', 'K21977', 'K22011', 'K22012', 'K22099', 'K22100', 'K22101', 'K22225', 'K22226', 'K22227', 'K22316', 'K22391', 'K22699', 'K22911', 'K22912', 'K22949', 'K23094', 'K23095', 'K23734', 'K23735', 'K23750', 'K23763', 'K24843', 'K24844', 'K24845', 'K24866', 'K25033', 'K25570', 'K28034', 'K28925', 'K28926'],
    # Part B | core_metabolism | ko01054
    'terpenoid_polyket': ['K01779', 'K15654', 'K15655', 'K15656', 'K15661', 'K15662', 'K15663', 'K15664', 'K15665', 'K15666', 'K15667', 'K15668', 'K16093', 'K16094', 'K16095', 'K16096', 'K16097', 'K16098', 'K16099', 'K16100', 'K16101', 'K16102', 'K16103', 'K16104', 'K16105', 'K16106', 'K16107', 'K16108', 'K16109', 'K16110', 'K16111', 'K16112', 'K16113', 'K16114', 'K16115', 'K16116', 'K16117', 'K16118', 'K16122', 'K16123', 'K16124', 'K16125', 'K16126', 'K16127', 'K16128', 'K16129', 'K16130', 'K16131', 'K16132', 'K16133', 'K16134', 'K26565', 'K26566', 'K26567', 'K26568', 'K26569', 'K26570', 'K26571', 'K26572', 'K26573', 'K26574', 'K26575', 'K26576', 'K26577', 'K26578', 'K26579'],
    # Part B | information_processing | ko02035
    'cell_motility': ['K00575', 'K02278', 'K02279', 'K02280', 'K02281', 'K02282', 'K02283', 'K02382', 'K02383', 'K02384', 'K02385', 'K02386', 'K02387', 'K02388', 'K02389', 'K02390', 'K02391', 'K02392', 'K02393', 'K02394', 'K02395', 'K02396', 'K02397', 'K02398', 'K02399', 'K02400', 'K02401', 'K02402', 'K02403', 'K02404', 'K02405', 'K02406', 'K02407', 'K02408', 'K02409', 'K02410', 'K02411', 'K02412', 'K02413', 'K02414', 'K02415', 'K02416', 'K02417', 'K02418', 'K02419', 'K02420', 'K02421', 'K02422', 'K02423', 'K02424', 'K02425', 'K02556', 'K02557', 'K02650', 'K02651', 'K02652', 'K02653', 'K02654', 'K02655', 'K02656', 'K02657', 'K02658', 'K02659', 'K02660', 'K02661', 'K02662', 'K02663', 'K02664', 'K02665', 'K02666', 'K02667', 'K02668', 'K02669', 'K02670', 'K02671', 'K02672', 'K02673', 'K02674', 'K02675', 'K02676', 'K03406', 'K03407', 'K03408', 'K03409', 'K03410', 'K03411', 'K03412', 'K03413', 'K03414', 'K03415', 'K03516', 'K03776', 'K04562', 'K05874', 'K05875', 'K05876', 'K05877', 'K06595', 'K06596', 'K06597', 'K06598', 'K06599', 'K06600', 'K06601', 'K06602', 'K06603', 'K06604', 'K07324', 'K07325', 'K07327', 'K07328', 'K07329', 'K07330', 'K07331', 'K07332', 'K07333', 'K07345', 'K07346', 'K07347', 'K07348', 'K07349', 'K07350', 'K07351', 'K07352', 'K07353', 'K07354', 'K07355', 'K07356', 'K07822', 'K07991', 'K09860', 'K10564', 'K10565', 'K10941', 'K10942', 'K10943', 'K11522', 'K11523', 'K11524', 'K11525', 'K11526', 'K13626', 'K13820', 'K13924', 'K18475', 'K21217', 'K21218', 'K23985', 'K23986', 'K24343', 'K24344', 'K24346', 'K27078'],
    # Part B | information_processing | ko04110
    'cell_growth_death': ['K02087', 'K02089', 'K02091', 'K02178', 'K02180', 'K02202', 'K02206', 'K02209', 'K02210', 'K02212', 'K02213', 'K02214', 'K02216', 'K02365', 'K02537', 'K02540', 'K02541', 'K02542', 'K02603', 'K02604', 'K02605', 'K02606', 'K02607', 'K02608', 'K03083', 'K03094', 'K03347', 'K03348', 'K03349', 'K03350', 'K03351', 'K03352', 'K03353', 'K03354', 'K03355', 'K03357', 'K03358', 'K03359', 'K03363', 'K03364', 'K03456', 'K03868', 'K03875', 'K04377', 'K04382', 'K04402', 'K04451', 'K04498', 'K04500', 'K04501', 'K04503', 'K04681', 'K04682', 'K04683', 'K04685', 'K04728', 'K04802', 'K05866', 'K05867', 'K05868', 'K06067', 'K06618', 'K06619', 'K06620', 'K06621', 'K06622', 'K06623', 'K06624', 'K06625', 'K06626', 'K06627', 'K06628', 'K06629', 'K06630', 'K06631', 'K06632', 'K06633', 'K06634', 'K06635', 'K06636', 'K06637', 'K06639', 'K06640', 'K06641', 'K06642', 'K06643', 'K06644', 'K06645', 'K06669', 'K06670', 'K06671', 'K06672', 'K06679', 'K08866', 'K09389', 'K09392', 'K09993', 'K10151', 'K10152', 'K10292', 'K10500', 'K10727', 'K10779', 'K11266', 'K11267', 'K11268', 'K11273', 'K11405', 'K11479', 'K11542', 'K11547', 'K11580', 'K11584', 'K12456', 'K13375', 'K13376', 'K13377', 'K13728', 'K16197', 'K16198', 'K16332', 'K17390', 'K17454', 'K21770', 'K21771', 'K22399', 'K23605', 'K25163', 'K25228', 'K25229', 'K26093', 'K26116', 'K26117'],
    # Part B | information_processing | ko03020
    'transcription': ['K00960', 'K02999', 'K03000', 'K03001', 'K03002', 'K03003', 'K03004', 'K03005', 'K03006', 'K03007', 'K03008', 'K03009', 'K03010', 'K03011', 'K03012', 'K03013', 'K03014', 'K03015', 'K03016', 'K03017', 'K03018', 'K03019', 'K03020', 'K03021', 'K03022', 'K03023', 'K03024', 'K03025', 'K03026', 'K03027', 'K03040', 'K03041', 'K03042', 'K03043', 'K03044', 'K03045', 'K03046', 'K03047', 'K03048', 'K03049', 'K03051', 'K03052', 'K03053', 'K03055', 'K03056', 'K03058', 'K03059', 'K03060', 'K13797', 'K13798', 'K14721', 'K16250', 'K16251', 'K16252', 'K16253', 'K21081', 'K21987', 'K25273', 'K25274', 'K25275', 'K25276', 'K25277', 'K25278', 'K25279', 'K25304', 'K25436'],
    # Part B | information_processing | ko03010
    'translation': ['K01977', 'K01979', 'K01980', 'K01981', 'K01982', 'K01984', 'K01985', 'K01986', 'K02863', 'K02864', 'K02865', 'K02866', 'K02867', 'K02868', 'K02869', 'K02870', 'K02871', 'K02872', 'K02873', 'K02874', 'K02875', 'K02876', 'K02877', 'K02878', 'K02879', 'K02880', 'K02881', 'K02882', 'K02883', 'K02884', 'K02885', 'K02886', 'K02887', 'K02888', 'K02889', 'K02890', 'K02891', 'K02892', 'K02893', 'K02894', 'K02895', 'K02896', 'K02897', 'K02898', 'K02899', 'K02900', 'K02901', 'K02902', 'K02903', 'K02904', 'K02905', 'K02906', 'K02907', 'K02908', 'K02909', 'K02910', 'K02911', 'K02912', 'K02913', 'K02914', 'K02915', 'K02916', 'K02917', 'K02918', 'K02919', 'K02920', 'K02921', 'K02922', 'K02923', 'K02924', 'K02925', 'K02926', 'K02927', 'K02928', 'K02929', 'K02930', 'K02931', 'K02932', 'K02933', 'K02934', 'K02935', 'K02936', 'K02937', 'K02938', 'K02939', 'K02940', 'K02941', 'K02942', 'K02943', 'K02944', 'K02945', 'K02946', 'K02947', 'K02948', 'K02949', 'K02950', 'K02951', 'K02952', 'K02953', 'K02954', 'K02955', 'K02956', 'K02957', 'K02958', 'K02959', 'K02960', 'K02961', 'K02962', 'K02963', 'K02964', 'K02965', 'K02966', 'K02967', 'K02968', 'K02969', 'K02970', 'K02971', 'K02973', 'K02974', 'K02975', 'K02976', 'K02977', 'K02978', 'K02979', 'K02980', 'K02981', 'K02982', 'K02983', 'K02984', 'K02985', 'K02986', 'K02987', 'K02988', 'K02989', 'K02990', 'K02991', 'K02992', 'K02993', 'K02994', 'K02995', 'K02996', 'K02997', 'K02998', 'K07590', 'K14753', 'K15033', 'K16174', 'K16830', 'K17401', 'K17402', 'K17403', 'K17404', 'K17405', 'K17406', 'K17407', 'K17408', 'K17409', 'K17410', 'K17411', 'K17412', 'K17413', 'K17414', 'K17415', 'K17417', 'K17418', 'K17419', 'K17420', 'K17421', 'K17422', 'K17423', 'K17424', 'K17425', 'K17426', 'K17427', 'K17428', 'K17429', 'K17430', 'K17431', 'K17432', 'K17433', 'K17434', 'K17435', 'K17436', 'K17437', 'K17438', 'K17439', 'K17440', 'K17659', 'K19032', 'K19033', 'K19034', 'K19035', 'K26191', 'K26192', 'K26193', 'K26194', 'K26195', 'K26196', 'K27290', 'K27351', 'K28346', 'K28351', 'K28397', 'K28431', 'K29076', 'K29077'],
    # Part B | information_processing | ko03060
    'protein_folding': ['K01551', 'K01983', 'K03070', 'K03071', 'K03072', 'K03073', 'K03074', 'K03075', 'K03076', 'K03100', 'K03101', 'K03104', 'K03105', 'K03106', 'K03107', 'K03108', 'K03109', 'K03110', 'K03116', 'K03117', 'K03118', 'K03210', 'K03217', 'K03425', 'K07342', 'K09481', 'K09490', 'K09540', 'K09647', 'K09648', 'K10956', 'K12257', 'K12272', 'K12275', 'K12946', 'K12947', 'K12948', 'K13280', 'K13301', 'K13431', 'K16365', 'K22384', 'K22386', 'K23387', 'K23388', 'K23390', 'K27163', 'K27164', 'K27165'],
    # Part B | information_processing | ko03030
    'replication_repair': ['K01972', 'K02209', 'K02210', 'K02212', 'K02314', 'K02316', 'K02319', 'K02320', 'K02321', 'K02322', 'K02323', 'K02324', 'K02325', 'K02326', 'K02327', 'K02328', 'K02335', 'K02337', 'K02338', 'K02339', 'K02340', 'K02341', 'K02342', 'K02343', 'K02344', 'K02345', 'K02540', 'K02541', 'K02542', 'K02683', 'K02684', 'K02685', 'K03111', 'K03469', 'K03470', 'K03471', 'K03504', 'K03505', 'K03506', 'K03763', 'K04799', 'K04800', 'K04801', 'K04802', 'K07466', 'K10726', 'K10739', 'K10740', 'K10741', 'K10742', 'K10743', 'K10744', 'K10745', 'K10747', 'K10754', 'K10755', 'K10756', 'K14159', 'K18882', 'K22316'],
    # Part B | metal_related | ko01501
    'amr': ['K00687', 'K01207', 'K01467', 'K02171', 'K02172', 'K02545', 'K02546', 'K02547', 'K03585', 'K03587', 'K03693', 'K05366', 'K05515', 'K08218', 'K08720', 'K09475', 'K09476', 'K10823', 'K12340', 'K12552', 'K12553', 'K12555', 'K12556', 'K15580', 'K15581', 'K15582', 'K15583', 'K17836', 'K17837', 'K17838', 'K17850', 'K18072', 'K18073', 'K18093', 'K18094', 'K18095', 'K18104', 'K18129', 'K18130', 'K18131', 'K18133', 'K18135', 'K18136', 'K18137', 'K18138', 'K18139', 'K18143', 'K18144', 'K18145', 'K18146', 'K18147', 'K18148', 'K18149', 'K18150', 'K18698', 'K18699', 'K18766', 'K18767', 'K18768', 'K18780', 'K18781', 'K18782', 'K18790', 'K18791', 'K18792', 'K18793', 'K18794', 'K18795', 'K18796', 'K18797', 'K18970', 'K18971', 'K18972', 'K18973', 'K18976', 'K19095', 'K19096', 'K19097', 'K19098', 'K19099', 'K19100', 'K19101', 'K19209', 'K19210', 'K19211', 'K19212', 'K19213', 'K19214', 'K19215', 'K19216', 'K19217', 'K19218', 'K19316', 'K19317', 'K19318', 'K19319', 'K19320', 'K19321', 'K19322', 'K20319', 'K20320', 'K21266', 'K21276', 'K21277', 'K22331', 'K22332', 'K22333', 'K22334', 'K22335', 'K22346', 'K22351', 'K22352'],
}

CATEGORY_META = {'sporulation': {'label': 'Sporulation', 'kegg': 'ko04111', 'group': 'negative_control', 'part': 'A'}, 'secondary_metab': {'label': 'Secondary metabolism', 'kegg': 'ko01110', 'group': 'negative_control', 'part': 'A'}, 'xenobiotics': {'label': 'Xenobiotics biodegradation', 'kegg': 'ko01120', 'group': 'negative_control', 'part': 'A'}, 'two_component': {'label': 'Two-component systems', 'kegg': 'ko02020', 'group': 'negative_control', 'part': 'A'}, 'abc_transporters': {'label': 'ABC transporters (non-metal)', 'kegg': 'ko02010−metal', 'group': 'negative_control', 'part': 'A'}, 'quorum_sensing': {'label': 'Quorum sensing', 'kegg': 'ko02024', 'group': 'negative_control', 'part': 'A'}, 'carbohydrate_metab': {'label': 'Carbohydrate metabolism', 'kegg': 'ko01200', 'group': 'core_metabolism', 'part': 'B'}, 'energy_metab': {'label': 'Energy metabolism', 'kegg': 'ko00190', 'group': 'core_metabolism', 'part': 'B'}, 'lipid_metab': {'label': 'Lipid metabolism', 'kegg': 'ko01212', 'group': 'core_metabolism', 'part': 'B'}, 'nucleotide_metab': {'label': 'Nucleotide metabolism', 'kegg': 'ko01232', 'group': 'core_metabolism', 'part': 'B'}, 'aa_metab': {'label': 'Amino acid metabolism', 'kegg': 'ko01230', 'group': 'core_metabolism', 'part': 'B'}, 'glycan_biosyn': {'label': 'Glycan biosynthesis (peptidoglycan)', 'kegg': 'ko00543', 'group': 'core_metabolism', 'part': 'B'}, 'cofactor_vitamin': {'label': 'Cofactors and vitamins', 'kegg': 'ko01240', 'group': 'core_metabolism', 'part': 'B'}, 'terpenoid_polyket': {'label': 'Terpenoids and polyketides', 'kegg': 'ko01054', 'group': 'core_metabolism', 'part': 'B'}, 'cell_motility': {'label': 'Cell motility', 'kegg': 'ko02035', 'group': 'information_processing', 'part': 'B'}, 'cell_growth_death': {'label': 'Cell growth and death', 'kegg': 'ko04110', 'group': 'information_processing', 'part': 'B'}, 'transcription': {'label': 'Transcription (RNA polymerase)', 'kegg': 'ko03020', 'group': 'information_processing', 'part': 'B'}, 'translation': {'label': 'Translation (ribosome)', 'kegg': 'ko03010', 'group': 'information_processing', 'part': 'B'}, 'protein_folding': {'label': 'Protein folding/degradation', 'kegg': 'ko03060', 'group': 'information_processing', 'part': 'B'}, 'replication_repair': {'label': 'Replication and repair', 'kegg': 'ko03030', 'group': 'information_processing', 'part': 'B'}, 'amr': {'label': 'Antimicrobial resistance (beta-lactam)', 'kegg': 'ko01501', 'group': 'metal_related', 'part': 'B'}}


# Verify KO counts
print(f"Total categories: {len(KEGG_CATEGORIES)}")
for name, kos in sorted(KEGG_CATEGORIES.items(), key=lambda x: len(x[1])):
    m = CATEGORY_META.get(name, {})
    print(f"  {name:35s}: {len(kos):5d} KOs  [{m.get('group','?')}]")



Total categories: 21
  protein_folding                    :    49 KOs  [information_processing]
  replication_repair                 :    60 KOs  [information_processing]
  terpenoid_polyket                  :    66 KOs  [core_metabolism]
  transcription                      :    66 KOs  [information_processing]
  glycan_biosyn                      :    73 KOs  [core_metabolism]
  lipid_metab                        :    84 KOs  [core_metabolism]
  amr                                :   112 KOs  [metal_related]
  nucleotide_metab                   :   123 KOs  [core_metabolism]
  sporulation                        :   126 KOs  [negative_control]
  cell_growth_death                  :   133 KOs  [information_processing]
  cell_motility                      :   153 KOs  [information_processing]
  translation                        :   206 KOs  [information_processing]
  energy_metab                       :   224 KOs  [core_metabolism]
  aa_metab                           :   242 KOs  [cor

In [3]:
## Block 2 — Spark density computation (identical to NB03 _spark_predictor)

def compute_ko_density(ko_ids_list):
    """Per-genus KO density (KOs per Mb) from kescience_mgnify.
    
    Returns DataFrame: genus_lower, ko_per_mb, n_mags
    """
    if not _SPARK_AVAILABLE:
        raise RuntimeError("Spark required — run in JupyterHub")

    ko_list = list(ko_ids_list)
    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)

    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (
            SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
            FROM kescience_mgnify.gene_eggnog
            WHERE kegg_ko IS NOT NULL AND kegg_ko != '-'
        ) koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb']   = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)
    gk = (pm.groupby('genus_lower', as_index=False)
            .agg(ko_per_mb=('ko_per_mb', 'mean'), n_mags=('genome_id', 'count')))
    return gk

def _z(s):
    return (s - s.mean()) / s.std()



In [4]:
## Block 3 — Run PGLS for all 21 categories

if not _SPARK_AVAILABLE:
    raise RuntimeError("Spark required for Block 3 — run in JupyterHub")

MIN_N = 100   # minimum genera required for a category to be included
results = []

for cat_key, ko_list in KEGG_CATEGORIES.items():
    meta = CATEGORY_META.get(cat_key, {})
    label_str = meta.get('label', cat_key)
    print(f"\n=== {label_str} ({len(ko_list)} KOs) ===")

    try:
        density_df = compute_ko_density(ko_list)
        density_df.to_csv(DATA / f'landscape_{cat_key}_density.csv', index=False)
        print(f"  Density: {len(density_df)} genera")
    except Exception as e:
        print(f"  Spark ERROR: {e}")
        results.append({'category': cat_key, 'description': label_str,
                        'n_genera': 0, 'n_KOs': len(ko_list), 'error': str(e)})
        continue

    merged = trait_df.merge(density_df[['genus_lower', 'ko_per_mb']],
                            on='genus_lower', how='inner').copy()
    merged['ko_per_mb_z'] = _z(merged['ko_per_mb'])
    n_avail = merged.dropna(subset=['ko_per_mb_z', 'mean_levins_B_std']).shape[0]
    print(f"  Joined genera: {n_avail}")

    if n_avail < MIN_N:
        print(f"  SKIPPED: n={n_avail} < {MIN_N}")
        results.append({'category': cat_key, 'description': label_str,
                        'n_genera': n_avail, 'n_KOs': len(ko_list),
                        'error': f'n<{MIN_N}'})
        continue

    try:
        res = run_pgls(
            merged.dropna(subset=['ko_per_mb_z', 'mean_levins_B_std']),
            TREE_BAC,
            response='mean_levins_B_std',
            predictors=['ko_per_mb_z'],
            taxon_col='genus_lower',
            label=f'FL_{cat_key}',
            min_n=MIN_N,
        )
        beta, SE, p, lam = res['beta'], res['SE'], res['p_value'], res['lambda_est']
        print(f"  β={beta:+.5f}, SE={SE:.5f}, p={p:.4g}, λ={lam:.4f}, n={res['n']}")
        results.append({
            'category':    cat_key,
            'description': label_str,
            'kegg_id':     meta.get('kegg', ''),
            'part':        meta.get('part', ''),
            'group':       meta.get('group', ''),
            'n_genera':    res['n'],
            'n_KOs':       len(ko_list),
            'lambda_est':  lam,
            'beta':        beta,
            'SE':          SE,
            'p_raw':       p,
            'r2':          res['r2'],
            'delta_aic':   res['delta_aic_vs_null'],
            'error':       '',
        })
    except Exception as exc:
        print(f"  PGLS ERROR: {exc}")
        results.append({'category': cat_key, 'description': label_str,
                        'n_genera': n_avail, 'n_KOs': len(ko_list), 'error': str(exc)})

print(f"\n=== Done: {len(results)} categories processed ===")




=== Sporulation (126 KOs) ===


  Density: 264 genera
  Joined genera: 26
  SKIPPED: n=26 < 100

=== Secondary metabolism (2326 KOs) ===


  Density: 10281 genera
  Joined genera: 1073


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.02763, SE=0.00423, p=9.884e-11, λ=0.8013, n=1073

=== Xenobiotics biodegradation (1449 KOs) ===


...

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.03485, SE=0.00484, p=1.072e-12, λ=0.7899, n=1073

=== Antimicrobial resistance (beta-lactam) (112 KOs) ===


  Density: 9843 genera
  Joined genera: 1073


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.00387, SE=0.00546, p=0.478, λ=0.8105, n=1073

=== Done: 21 categories processed ===


In [5]:
## Block 4 — BH-FDR correction and save CSV

res_df = pd.DataFrame(results)
valid  = res_df[res_df['error'].fillna('') == ''].copy()
skipped = res_df[res_df['error'].fillna('') != '']

if len(skipped):
    print("Skipped categories:")
    for _, r in skipped.iterrows():
        print(f"  {r['category']:30s}: {r.get('error','')}")

# BH-FDR across all valid categories
if len(valid):
    _, q_bh, _, _ = multipletests(valid['p_raw'].values, method='fdr_bh')
    valid = valid.copy()
    valid['q_bh'] = q_bh

    # Add the P1 metal reference row
    metal_ref = pd.DataFrame([{
        'category':    'metal_genes_p1',
        'description': 'Metal genes Tier 1+2 (P1 reference)',
        'kegg_id':     'curated_140KO',
        'part':        'REF',
        'group':       'metal_reference',
        'n_genera':    1574,
        'n_KOs':       140,
        'lambda_est':  0.757,
        'beta':        -0.021,
        'SE':          0.0037,
        'p_raw':       2.1e-8,
        'q_bh':        float('nan'),   # not part of this FDR family
        'r2':          float('nan'),
        'delta_aic':   float('nan'),
        'error':       '',
    }])
    out_df = pd.concat([valid, metal_ref], ignore_index=True)
else:
    out_df = res_df

# Rename for output
out_df = out_df.rename(columns={'r2': 'partial_R2', 'p_raw': 'p_raw', 'q_bh': 'q_bh'})
out_df.to_csv(DATA / 'functional_landscape_results.csv', index=False)
print(f"Saved: data/functional_landscape_results.csv ({len(out_df)} rows)")
print()
print("Results (valid, sorted by β):")
for _, r in out_df[out_df['error'].fillna('') == ''].sort_values('beta').iterrows():
    sig = '***' if r.get('q_bh', float('nan')) <= 0.05 else '   '
    print(f"  {sig} {r['description']:45s}: β={r['beta']:+.4f}  q={r.get('q_bh', float('nan')):.3f}  n={int(r['n_genera'])}")



Skipped categories:
  sporulation                   : n<100
  cell_growth_death             : n<100
Saved: data/functional_landscape_results.csv (20 rows)

Results (valid, sorted by β):
  *** Replication and repair                       : β=-0.0349  q=0.000  n=1073
  *** Nucleotide metabolism                        : β=-0.0321  q=0.000  n=1073
  *** Amino acid metabolism                        : β=-0.0306  q=0.000  n=1073
  *** Translation (ribosome)                       : β=-0.0299  q=0.000  n=1073
  *** Protein folding/degradation                  : β=-0.0296  q=0.000  n=1073
  *** Cofactors and vitamins                       : β=-0.0292  q=0.000  n=1073
  *** Secondary metabolism                         : β=-0.0276  q=0.000  n=1073
  *** Transcription (RNA polymerase)               : β=-0.0276  q=0.000  n=1071
  *** Carbohydrate metabolism                      : β=-0.0263  q=0.000  n=1073
  *** Lipid metabolism                             : β=-0.0211  q=0.000  n=1069
      Metal ge

In [6]:
## Block 5 — Forest plot: functional landscape

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

res_df = pd.read_csv(DATA / 'functional_landscape_results.csv')
valid  = res_df[res_df['error'].fillna('') == ''].copy()
valid  = valid.sort_values('beta', ascending=False).reset_index(drop=True)

# Colours
colors = [GROUP_COLORS.get(g, '#999999') for g in valid['group'].fillna('unknown')]

fig_h = max(8, len(valid) * 0.45)
fig, ax = plt.subplots(figsize=(11, fig_h))

for i, (_, row) in enumerate(valid.iterrows()):
    y = len(valid) - 1 - i
    b, se = float(row['beta']), float(row['SE'])
    ci = 1.96 * se
    color = GROUP_COLORS.get(row.get('group',''), '#999999')

    # Bar / error bar
    ax.errorbar(b, y, xerr=ci, fmt='none', color=color, elinewidth=1.8,
                capsize=4, capthick=1.8, zorder=3)
    marker = 's' if row.get('group') == 'metal_reference' else 'o'
    mfc = color if row.get('q_bh', 1.0) <= 0.05 or row.get('group') == 'metal_reference' else 'white'
    ax.plot(b, y, marker=marker, color=color, markerfacecolor=mfc,
            markersize=8, markeredgewidth=1.8, zorder=4)

    # Annotation: n, n_KOs, q
    q_val = row.get('q_bh', float('nan'))
    q_str = f"q={q_val:.3f}" if pd.notna(q_val) and q_val != '' else ''
    ann = f"n={int(row['n_genera'])}, {int(row['n_KOs'])} KOs  {q_str}"
    ax.text(ax.get_xlim()[1] if ax.get_xlim()[1] > 0 else 0.01, y,
            ann, va='center', ha='left', fontsize=6.5, color='#444444')

ax.axvline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.7, zorder=2)

# Y-axis labels
ytick_labels = valid['description'].tolist()
ytick_labels = [ytick_labels[len(valid)-1-i] for i in range(len(valid))]
ax.set_yticks(range(len(valid)))
ax.set_yticklabels([valid.loc[len(valid)-1-i, 'description'] for i in range(len(valid))],
                   fontsize=8.5)

ax.set_xlabel('PGLS β ± 95% CI (ko_per_mb_z → niche breadth)', fontsize=10)
ax.set_title('Functional landscape: per-Mb gene density vs. niche breadth\n'
             'across 21 KEGG functional categories', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
legend_handles = [mpatches.Patch(color=c, label=l) for g, (c, l) in
                  {g: (c, GROUP_LABELS[g]) for g, c in GROUP_COLORS.items()}.items()]
ax.legend(handles=legend_handles, fontsize=8, loc='lower right',
          framealpha=0.9, edgecolor='#cccccc')

plt.tight_layout()
fig.savefig(str(FIGS / 'functional_landscape_forest.png'), dpi=300, bbox_inches='tight')
plt.close()
print("Saved: figures/functional_landscape_forest.png")



Saved: figures/functional_landscape_forest.png


In [7]:
## Block 6 — Print summary for REPORT.md

res_df = pd.read_csv(DATA / 'functional_landscape_results.csv')
valid  = res_df[res_df['error'].fillna('') == ''].sort_values('beta')

print("\n=== FUNCTIONAL LANDSCAPE SUMMARY ===")
print(f"\nCategories tested: {len(valid)}")
print(f"FDR-significant (q ≤ 0.05): {(valid['q_bh'] <= 0.05).sum()}")
print(f"β range: {valid['beta'].min():.4f} to {valid['beta'].max():.4f}")
print(f"Metal P1 reference: β = −0.021")
print()

print("Strongest negative associations (top 5 by |β|, β<0):")
for _, r in valid[valid['beta'] < 0].head(5).iterrows():
    print(f"  {r['description']:40s}: β={r['beta']:+.4f}  q={r.get('q_bh', float('nan')):.4f}")

print("\nWeakest / null associations (|β| < 0.005):")
near_null = valid[valid['beta'].abs() < 0.005]
for _, r in near_null.iterrows():
    print(f"  {r['description']:40s}: β={r['beta']:+.4f}  q={r.get('q_bh', float('nan')):.4f}")

print("\nPositive associations (β > 0):")
pos = valid[valid['beta'] > 0]
if len(pos):
    for _, r in pos.iterrows():
        print(f"  {r['description']:40s}: β={r['beta']:+.4f}  q={r.get('q_bh', float('nan')):.4f}")
else:
    print("  None")




=== FUNCTIONAL LANDSCAPE SUMMARY ===

Categories tested: 20
FDR-significant (q ≤ 0.05): 14
β range: -0.0349 to 0.0059
Metal P1 reference: β = −0.021

Strongest negative associations (top 5 by |β|, β<0):
  Replication and repair                  : β=-0.0349  q=0.0000
  Nucleotide metabolism                   : β=-0.0321  q=0.0000
  Amino acid metabolism                   : β=-0.0306  q=0.0000
  Translation (ribosome)                  : β=-0.0299  q=0.0000
  Protein folding/degradation             : β=-0.0296  q=0.0000

Weakest / null associations (|β| < 0.005):
  Antimicrobial resistance (beta-lactam)  : β=-0.0039  q=0.5046
  Glycan biosynthesis (peptidoglycan)     : β=+0.0012  q=0.7667
  Cell motility                           : β=+0.0040  q=0.5046

Positive associations (β > 0):
  Glycan biosynthesis (peptidoglycan)     : β=+0.0012  q=0.7667
  Cell motility                           : β=+0.0040  q=0.5046
  Two-component systems                   : β=+0.0059  q=0.4569
